# CURE-OR++ Open-Weight VLM Real-Transfer GPU Matrix

This notebook runs open-weight VLMs over a CURE-OR++ real-transfer prompt pack.
It uses a tiered model matrix so we can smoke-test multiple models before
promoting stable rows to full evaluation.

Default mode: smoke test enabled models from
the configured model matrix.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tarfile
import zipfile
import pandas as pd

os.environ.setdefault('HF_HOME', '/tmp/cure_or_pp_hf_home')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# Generated by scripts/write_kaggle_vlm_notebook.py.
RUN_MODE = 'smoke'  # 'smoke' or 'full'
SELECTED_MODEL_SLUGS = ["llava_onevision_qwen2_7b", "qwen2_5_vl_7b"]  # [] means enabled_by_default models from the matrix.
EMBEDDED_RUNTIME_FILES = {"configs/vlm_open_weight_model_matrix_v03.json": "{\n  \"matrix_id\": \"vlm_open_weight_real_transfer_v03\",\n  \"description\": \"Tiered open-weight VLM matrix for the CURE-OR++ real-transfer v0.3 prompt pack. Use smoke mode before full 900-row runs.\",\n  \"prompt_pack_path\": \"reports/vlm_api_track_v03_prompt_pack.jsonl\",\n  \"default_run_mode\": \"smoke\",\n  \"smoke_limit\": 6,\n  \"smoke_sample_ids\": [\n    \"clean__clean__clean__09904\",\n    \"clean__clean__clean__10183\",\n    \"real_transfer__messenger_upload_download__rep_01__10183\",\n    \"real_transfer__phone_screenshot_resave__rep_01__10183\",\n    \"real_transfer__video_call_frame_capture__rep_01__10183\",\n    \"real_transfer__social_app_resave__rep_01__10183\"\n  ],\n  \"full_limit\": 900,\n  \"selection_policy\": \"Run memory-controlled completed v0.2 models in smoke mode first. Promote successful rows to full 900-row v0.3 runs. Start with llava_onevision_qwen2_7b, qwen2_5_vl_7b, smolvlm2_2b, and open-weight family contrast rows.\",\n  \"models\": [\n    {\n      \"slug\": \"smolvlm2_2b\",\n      \"model_id\": \"HuggingFaceTB/SmolVLM2-2.2B-Instruct\",\n      \"provider\": \"huggingface\",\n      \"model_class\": \"AutoModelForImageTextToText\",\n      \"dtype\": \"float16\",\n      \"image_content_key\": \"path\",\n      \"max_tokens\": 8,\n      \"tier\": \"v03_priority\",\n      \"enabled_by_default\": false,\n      \"status\": \"v02_full_completed_v03_pending\",\n      \"known_result_dir\": \"reports/vlm_open_weight_smolvlm2_2b_kaggle_v01\",\n      \"license\": \"apache-2.0\",\n      \"hf_url\": \"https://huggingface.co/HuggingFaceTB/SmolVLM2-2.2B-Instruct\"\n    },\n    {\n      \"slug\": \"internvl3_1b\",\n      \"model_id\": \"OpenGVLab/InternVL3-1B-hf\",\n      \"provider\": \"huggingface\",\n      \"model_class\": \"AutoModelForImageTextToText\",\n      \"dtype\": \"float16\",\n      \"image_content_key\": \"url_path\",\n      \"max_tokens\": 8,\n      \"tier\": \"v03_family_contrast\",\n      \"enabled_by_default\": false,\n      \"status\": \"v02_full_completed_v03_pending\",\n      \"known_result_dir\": \"reports/vlm_open_weight_internvl3_1b_kaggle_v01\",\n      \"license\": \"qwen\",\n      \"hf_url\": \"https://huggingface.co/OpenGVLab/InternVL3-1B-hf\"\n    },\n    {\n      \"slug\": \"internvl3_2b\",\n      \"model_id\": \"OpenGVLab/InternVL3-2B-hf\",\n      \"provider\": \"huggingface\",\n      \"model_class\": \"AutoModelForImageTextToText\",\n      \"dtype\": \"float16\",\n      \"image_content_key\": \"url_path\",\n      \"max_tokens\": 8,\n      \"tier\": \"v03_family_contrast\",\n      \"enabled_by_default\": false,\n      \"status\": \"v02_full_completed_v03_pending\",\n      \"known_result_dir\": \"reports/vlm_open_weight_internvl3_2b_kaggle_v01\",\n      \"license\": \"qwen\",\n      \"hf_url\": \"https://huggingface.co/OpenGVLab/InternVL3-2B-hf\"\n    },\n    {\n      \"slug\": \"qwen2_5_vl_3b\",\n      \"model_id\": \"Qwen/Qwen2.5-VL-3B-Instruct\",\n      \"provider\": \"huggingface\",\n      \"model_class\": \"Qwen2_5_VLForConditionalGeneration\",\n      \"generation_backend\": \"qwen_vl_utils\",\n      \"dtype\": \"float16\",\n      \"image_content_key\": \"image\",\n      \"processor_kwargs\": {\n        \"min_pixels\": 200704,\n        \"max_pixels\": 1003520\n      },\n      \"max_tokens\": 8,\n      \"tier\": \"v03_priority\",\n      \"enabled_by_default\": false,\n      \"status\": \"v02_full_completed_v03_pending\",\n      \"known_result_dir\": \"reports/vlm_open_weight_qwen2_5_vl_3b_kaggle_v01\",\n      \"license\": \"unknown_from_hf_header\",\n      \"hf_url\": \"https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct\"\n    },\n    {\n      \"slug\": \"qwen2_5_vl_7b\",\n      \"model_id\": \"Qwen/Qwen2.5-VL-7B-Instruct\",\n      \"provider\": \"huggingface\",\n      \"model_class\": \"Qwen2_5_VLForConditionalGeneration\",\n      \"generation_backend\": \"qwen_vl_utils\",\n      \"dtype\": \"float16\",\n      \"image_content_key\": \"image\",\n      \"model_kwargs\": {\n        \"low_cpu_mem_usage\": true\n      },\n      \"processor_kwargs\": {\n        \"min_pixels\": 50176,\n        \"max_pixels\": 262144\n      },\n      \"generate_kwargs\": {\n        \"use_cache\": false\n      },\n      \"image_max_side\": 512,\n      \"max_tokens\": 8,\n      \"tier\": \"v03_priority\",\n      \"enabled_by_default\": false,\n      \"status\": \"v02_full_completed_v03_pending\",\n      \"known_result_dir\": \"reports/vlm_open_weight_qwen2_5_vl_7b_kaggle_v01\",\n      \"license\": \"apache-2.0\",\n      \"hf_url\": \"https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct\"\n    },\n    {\n      \"slug\": \"llava_onevision_qwen2_0_5b\",\n      \"model_id\": \"llava-hf/llava-onevision-qwen2-0.5b-ov-hf\",\n      \"provider\": \"huggingface\",\n      \"model_class\": \"LlavaOnevisionForConditionalGeneration\",\n      \"dtype\": \"float16\",\n      \"image_content_key\": \"url_path\",\n      \"model_kwargs\": {\n        \"low_cpu_mem_usage\": true\n      },\n      \"generate_kwargs\": {\n        \"use_cache\": false\n      },\n      \"device_map\": \"auto\",\n      \"image_max_side\": 768,\n      \"max_tokens\": 8,\n      \"tier\": \"v03_family_contrast\",\n      \"enabled_by_default\": false,\n      \"status\": \"v02_full_completed_v03_pending\",\n      \"known_result_dir\": \"reports/vlm_open_weight_llava_onevision_qwen2_0_5b_kaggle_v01\",\n      \"license\": \"apache-2.0\",\n      \"hf_url\": \"https://huggingface.co/llava-hf/llava-onevision-qwen2-0.5b-ov-hf\"\n    },\n    {\n      \"slug\": \"llava_onevision_qwen2_7b\",\n      \"model_id\": \"llava-hf/llava-onevision-qwen2-7b-ov-hf\",\n      \"provider\": \"huggingface\",\n      \"model_class\": \"LlavaOnevisionForConditionalGeneration\",\n      \"dtype\": \"float16\",\n      \"image_content_key\": \"url_path\",\n      \"model_kwargs\": {\n        \"low_cpu_mem_usage\": true\n      },\n      \"generate_kwargs\": {\n        \"use_cache\": false\n      },\n      \"device_map\": \"auto\",\n      \"image_max_side\": 384,\n      \"max_tokens\": 2,\n      \"tier\": \"v03_priority\",\n      \"enabled_by_default\": false,\n      \"status\": \"v02_full_completed_v03_pending\",\n      \"known_result_dir\": \"reports/vlm_open_weight_llava_onevision_qwen2_7b_kaggle_v01\",\n      \"known_smoke_result_dir\": \"reports/vlm_open_weight_llava_onevision_qwen2_7b_kaggle_smoke_v01\",\n      \"license\": \"apache-2.0\",\n      \"hf_url\": \"https://huggingface.co/llava-hf/llava-onevision-qwen2-7b-ov-hf\"\n    }\n  ]\n}\n", "reports/vlm_api_track_v03_prompt_pack.jsonl": "{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/09904.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__09904\", \"source_path\": \"data/raw/mini_cure_or/test/09904.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_01\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/09958.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__09958\", \"source_path\": \"data/raw/mini_cure_or/test/09958.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_01\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/10183.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__10183\", \"source_path\": \"data/raw/mini_cure_or/test/10183.jpg\", \"source_selection_id\": \"rtv03_baseball_01\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/10243.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__10243\", \"source_path\": \"data/raw/mini_cure_or/test/10243.jpg\", \"source_selection_id\": \"rtv03_canon_camera_01\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/10457.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__10457\", \"source_path\": \"data/raw/mini_cure_or/test/10457.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_01\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/10530.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__10530\", \"source_path\": \"data/raw/mini_cure_or/test/10530.jpg\", \"source_selection_id\": \"rtv03_canon_camera_02\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/10640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__10640\", \"source_path\": \"data/raw/mini_cure_or/test/10640.jpg\", \"source_selection_id\": \"rtv03_shoes_01\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/10714.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__10714\", \"source_path\": \"data/raw/mini_cure_or/test/10714.jpg\", \"source_selection_id\": \"rtv03_pan_01\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/10854.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__10854\", \"source_path\": \"data/raw/mini_cure_or/test/10854.jpg\", \"source_selection_id\": \"rtv03_shoes_02\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11059.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11059\", \"source_path\": \"data/raw/mini_cure_or/test/11059.jpg\", \"source_selection_id\": \"rtv03_baseball_02\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11101.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11101\", \"source_path\": \"data/raw/mini_cure_or/test/11101.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_02\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11247.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11247\", \"source_path\": \"data/raw/mini_cure_or/test/11247.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_01\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11285.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11285\", \"source_path\": \"data/raw/mini_cure_or/test/11285.jpg\", \"source_selection_id\": \"rtv03_pan_02\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11291.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11291\", \"source_path\": \"data/raw/mini_cure_or/test/11291.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_03\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11335.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11335\", \"source_path\": \"data/raw/mini_cure_or/test/11335.jpg\", \"source_selection_id\": \"rtv03_hair_brush_01\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11437.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11437\", \"source_path\": \"data/raw/mini_cure_or/test/11437.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_02\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11442.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11442\", \"source_path\": \"data/raw/mini_cure_or/test/11442.jpg\", \"source_selection_id\": \"rtv03_baseball_03\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11446.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11446\", \"source_path\": \"data/raw/mini_cure_or/test/11446.jpg\", \"source_selection_id\": \"rtv03_shoes_03\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11460.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11460\", \"source_path\": \"data/raw/mini_cure_or/test/11460.jpg\", \"source_selection_id\": \"rtv03_pan_03\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11608.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11608\", \"source_path\": \"data/raw/mini_cure_or/test/11608.jpg\", \"source_selection_id\": \"rtv03_pan_04\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11677.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11677\", \"source_path\": \"data/raw/mini_cure_or/test/11677.jpg\", \"source_selection_id\": \"rtv03_baseball_04\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11716.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11716\", \"source_path\": \"data/raw/mini_cure_or/test/11716.jpg\", \"source_selection_id\": \"rtv03_toy_01\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11735.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11735\", \"source_path\": \"data/raw/mini_cure_or/test/11735.jpg\", \"source_selection_id\": \"rtv03_toy_02\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11747.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11747\", \"source_path\": \"data/raw/mini_cure_or/test/11747.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_03\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11758.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11758\", \"source_path\": \"data/raw/mini_cure_or/test/11758.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_04\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11887.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11887\", \"source_path\": \"data/raw/mini_cure_or/test/11887.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_02\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/11890.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__11890\", \"source_path\": \"data/raw/mini_cure_or/test/11890.jpg\", \"source_selection_id\": \"rtv03_hair_brush_02\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/12027.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__12027\", \"source_path\": \"data/raw/mini_cure_or/test/12027.jpg\", \"source_selection_id\": \"rtv03_toy_03\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/12043.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__12043\", \"source_path\": \"data/raw/mini_cure_or/test/12043.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_03\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/12528.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__12528\", \"source_path\": \"data/raw/mini_cure_or/test/12528.jpg\", \"source_selection_id\": \"rtv03_canon_camera_03\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/12550.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__12550\", \"source_path\": \"data/raw/mini_cure_or/test/12550.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_05\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/12582.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__12582\", \"source_path\": \"data/raw/mini_cure_or/test/12582.jpg\", \"source_selection_id\": \"rtv03_pan_05\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/12605.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__12605\", \"source_path\": \"data/raw/mini_cure_or/test/12605.jpg\", \"source_selection_id\": \"rtv03_shoes_04\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/12704.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__12704\", \"source_path\": \"data/raw/mini_cure_or/test/12704.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_06\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/12706.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__12706\", \"source_path\": \"data/raw/mini_cure_or/test/12706.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_04\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/12815.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__12815\", \"source_path\": \"data/raw/mini_cure_or/test/12815.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_02\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/12844.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__12844\", \"source_path\": \"data/raw/mini_cure_or/test/12844.jpg\", \"source_selection_id\": \"rtv03_pan_06\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13037.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13037\", \"source_path\": \"data/raw/mini_cure_or/test/13037.jpg\", \"source_selection_id\": \"rtv03_shoes_05\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13124.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13124\", \"source_path\": \"data/raw/mini_cure_or/test/13124.jpg\", \"source_selection_id\": \"rtv03_canon_camera_04\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13142.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13142\", \"source_path\": \"data/raw/mini_cure_or/test/13142.jpg\", \"source_selection_id\": \"rtv03_pan_07\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13252.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13252\", \"source_path\": \"data/raw/mini_cure_or/test/13252.jpg\", \"source_selection_id\": \"rtv03_shoes_06\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13291.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13291\", \"source_path\": \"data/raw/mini_cure_or/test/13291.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_05\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13315.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13315\", \"source_path\": \"data/raw/mini_cure_or/test/13315.jpg\", \"source_selection_id\": \"rtv03_hair_brush_03\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13506.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13506\", \"source_path\": \"data/raw/mini_cure_or/test/13506.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_03\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13590.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13590\", \"source_path\": \"data/raw/mini_cure_or/test/13590.jpg\", \"source_selection_id\": \"rtv03_baseball_05\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13638.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13638\", \"source_path\": \"data/raw/mini_cure_or/test/13638.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_04\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13641.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13641\", \"source_path\": \"data/raw/mini_cure_or/test/13641.jpg\", \"source_selection_id\": \"rtv03_canon_camera_05\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13712.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13712\", \"source_path\": \"data/raw/mini_cure_or/test/13712.jpg\", \"source_selection_id\": \"rtv03_hair_brush_04\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13724.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13724\", \"source_path\": \"data/raw/mini_cure_or/test/13724.jpg\", \"source_selection_id\": \"rtv03_baseball_06\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13725.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13725\", \"source_path\": \"data/raw/mini_cure_or/test/13725.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_06\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13780.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13780\", \"source_path\": \"data/raw/mini_cure_or/test/13780.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_07\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13819.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13819\", \"source_path\": \"data/raw/mini_cure_or/test/13819.jpg\", \"source_selection_id\": \"rtv03_shoes_07\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13820.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13820\", \"source_path\": \"data/raw/mini_cure_or/test/13820.jpg\", \"source_selection_id\": \"rtv03_hair_brush_05\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13864.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13864\", \"source_path\": \"data/raw/mini_cure_or/test/13864.jpg\", \"source_selection_id\": \"rtv03_toy_04\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13870.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13870\", \"source_path\": \"data/raw/mini_cure_or/test/13870.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_08\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13897.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13897\", \"source_path\": \"data/raw/mini_cure_or/test/13897.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_09\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13924.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13924\", \"source_path\": \"data/raw/mini_cure_or/test/13924.jpg\", \"source_selection_id\": \"rtv03_canon_camera_06\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/13931.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__13931\", \"source_path\": \"data/raw/mini_cure_or/test/13931.jpg\", \"source_selection_id\": \"rtv03_toy_05\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/14003.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__14003\", \"source_path\": \"data/raw/mini_cure_or/test/14003.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_05\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/14029.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__14029\", \"source_path\": \"data/raw/mini_cure_or/test/14029.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_06\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/14220.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__14220\", \"source_path\": \"data/raw/mini_cure_or/test/14220.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_04\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/14382.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__14382\", \"source_path\": \"data/raw/mini_cure_or/test/14382.jpg\", \"source_selection_id\": \"rtv03_pan_08\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/14405.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__14405\", \"source_path\": \"data/raw/mini_cure_or/test/14405.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_05\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/14426.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__14426\", \"source_path\": \"data/raw/mini_cure_or/test/14426.jpg\", \"source_selection_id\": \"rtv03_shoes_08\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/14484.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__14484\", \"source_path\": \"data/raw/mini_cure_or/test/14484.jpg\", \"source_selection_id\": \"rtv03_canon_camera_07\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/14640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__14640\", \"source_path\": \"data/raw/mini_cure_or/test/14640.jpg\", \"source_selection_id\": \"rtv03_shoes_09\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/14751.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__14751\", \"source_path\": \"data/raw/mini_cure_or/test/14751.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_07\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/14754.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__14754\", \"source_path\": \"data/raw/mini_cure_or/test/14754.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_07\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/14773.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__14773\", \"source_path\": \"data/raw/mini_cure_or/test/14773.jpg\", \"source_selection_id\": \"rtv03_baseball_07\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/14836.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__14836\", \"source_path\": \"data/raw/mini_cure_or/test/14836.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_08\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/14901.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__14901\", \"source_path\": \"data/raw/mini_cure_or/test/14901.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_08\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/14978.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__14978\", \"source_path\": \"data/raw/mini_cure_or/test/14978.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_06\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15012.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15012\", \"source_path\": \"data/raw/mini_cure_or/test/15012.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_07\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15024.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15024\", \"source_path\": \"data/raw/mini_cure_or/test/15024.jpg\", \"source_selection_id\": \"rtv03_baseball_08\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15051.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15051\", \"source_path\": \"data/raw/mini_cure_or/test/15051.jpg\", \"source_selection_id\": \"rtv03_hair_brush_06\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15150.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15150\", \"source_path\": \"data/raw/mini_cure_or/test/15150.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_09\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15162.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15162\", \"source_path\": \"data/raw/mini_cure_or/test/15162.jpg\", \"source_selection_id\": \"rtv03_hair_brush_07\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15164.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15164\", \"source_path\": \"data/raw/mini_cure_or/test/15164.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_10\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15207.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15207\", \"source_path\": \"data/raw/mini_cure_or/test/15207.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_08\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15222.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15222\", \"source_path\": \"data/raw/mini_cure_or/test/15222.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_09\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15247.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15247\", \"source_path\": \"data/raw/mini_cure_or/test/15247.jpg\", \"source_selection_id\": \"rtv03_toy_06\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15309.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15309\", \"source_path\": \"data/raw/mini_cure_or/test/15309.jpg\", \"source_selection_id\": \"rtv03_baseball_09\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15321.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15321\", \"source_path\": \"data/raw/mini_cure_or/test/15321.jpg\", \"source_selection_id\": \"rtv03_toy_07\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15381.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15381\", \"source_path\": \"data/raw/mini_cure_or/test/15381.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_10\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15409.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15409\", \"source_path\": \"data/raw/mini_cure_or/test/15409.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_10\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15491.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15491\", \"source_path\": \"data/raw/mini_cure_or/test/15491.jpg\", \"source_selection_id\": \"rtv03_hair_brush_08\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15550.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15550\", \"source_path\": \"data/raw/mini_cure_or/test/15550.jpg\", \"source_selection_id\": \"rtv03_hair_brush_09\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15600.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15600\", \"source_path\": \"data/raw/mini_cure_or/test/15600.jpg\", \"source_selection_id\": \"rtv03_toy_08\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15604.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15604\", \"source_path\": \"data/raw/mini_cure_or/test/15604.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_09\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15621.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15621\", \"source_path\": \"data/raw/mini_cure_or/test/15621.jpg\", \"source_selection_id\": \"rtv03_baseball_10\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15631.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15631\", \"source_path\": \"data/raw/mini_cure_or/test/15631.jpg\", \"source_selection_id\": \"rtv03_shoes_10\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15716.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15716\", \"source_path\": \"data/raw/mini_cure_or/test/15716.jpg\", \"source_selection_id\": \"rtv03_canon_camera_08\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15772.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15772\", \"source_path\": \"data/raw/mini_cure_or/test/15772.jpg\", \"source_selection_id\": \"rtv03_pan_09\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15808.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15808\", \"source_path\": \"data/raw/mini_cure_or/test/15808.jpg\", \"source_selection_id\": \"rtv03_pan_10\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15874.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15874\", \"source_path\": \"data/raw/mini_cure_or/test/15874.jpg\", \"source_selection_id\": \"rtv03_hair_brush_10\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/15877.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__15877\", \"source_path\": \"data/raw/mini_cure_or/test/15877.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_10\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/16118.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__16118\", \"source_path\": \"data/raw/mini_cure_or/test/16118.jpg\", \"source_selection_id\": \"rtv03_toy_09\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/16451.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__16451\", \"source_path\": \"data/raw/mini_cure_or/test/16451.jpg\", \"source_selection_id\": \"rtv03_canon_camera_09\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/16481.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__16481\", \"source_path\": \"data/raw/mini_cure_or/test/16481.jpg\", \"source_selection_id\": \"rtv03_canon_camera_10\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"clean\", \"image_path\": \"data/raw/mini_cure_or/test/16499.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"clean\", \"repeat_id\": \"clean\", \"sample_id\": \"clean__clean__clean__16499\", \"source_path\": \"data/raw/mini_cure_or/test/16499.jpg\", \"source_selection_id\": \"rtv03_toy_10\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/10183.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__10183\", \"source_path\": \"data/raw/mini_cure_or/test/10183.jpg\", \"source_selection_id\": \"rtv03_baseball_01\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/10183.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__10183\", \"source_path\": \"data/raw/mini_cure_or/test/10183.jpg\", \"source_selection_id\": \"rtv03_baseball_01\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/10183.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__10183\", \"source_path\": \"data/raw/mini_cure_or/test/10183.jpg\", \"source_selection_id\": \"rtv03_baseball_01\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/10183.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__10183\", \"source_path\": \"data/raw/mini_cure_or/test/10183.jpg\", \"source_selection_id\": \"rtv03_baseball_01\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/10183.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__10183\", \"source_path\": \"data/raw/mini_cure_or/test/10183.jpg\", \"source_selection_id\": \"rtv03_baseball_01\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/10183.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__10183\", \"source_path\": \"data/raw/mini_cure_or/test/10183.jpg\", \"source_selection_id\": \"rtv03_baseball_01\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/10183.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__10183\", \"source_path\": \"data/raw/mini_cure_or/test/10183.jpg\", \"source_selection_id\": \"rtv03_baseball_01\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/10183.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__10183\", \"source_path\": \"data/raw/mini_cure_or/test/10183.jpg\", \"source_selection_id\": \"rtv03_baseball_01\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11059.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11059\", \"source_path\": \"data/raw/mini_cure_or/test/11059.jpg\", \"source_selection_id\": \"rtv03_baseball_02\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11059.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11059\", \"source_path\": \"data/raw/mini_cure_or/test/11059.jpg\", \"source_selection_id\": \"rtv03_baseball_02\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11059.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11059\", \"source_path\": \"data/raw/mini_cure_or/test/11059.jpg\", \"source_selection_id\": \"rtv03_baseball_02\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11059.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11059\", \"source_path\": \"data/raw/mini_cure_or/test/11059.jpg\", \"source_selection_id\": \"rtv03_baseball_02\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11059.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11059\", \"source_path\": \"data/raw/mini_cure_or/test/11059.jpg\", \"source_selection_id\": \"rtv03_baseball_02\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11059.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11059\", \"source_path\": \"data/raw/mini_cure_or/test/11059.jpg\", \"source_selection_id\": \"rtv03_baseball_02\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11059.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11059\", \"source_path\": \"data/raw/mini_cure_or/test/11059.jpg\", \"source_selection_id\": \"rtv03_baseball_02\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11059.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11059\", \"source_path\": \"data/raw/mini_cure_or/test/11059.jpg\", \"source_selection_id\": \"rtv03_baseball_02\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11442.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11442\", \"source_path\": \"data/raw/mini_cure_or/test/11442.jpg\", \"source_selection_id\": \"rtv03_baseball_03\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11442.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11442\", \"source_path\": \"data/raw/mini_cure_or/test/11442.jpg\", \"source_selection_id\": \"rtv03_baseball_03\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11442.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11442\", \"source_path\": \"data/raw/mini_cure_or/test/11442.jpg\", \"source_selection_id\": \"rtv03_baseball_03\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11442.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11442\", \"source_path\": \"data/raw/mini_cure_or/test/11442.jpg\", \"source_selection_id\": \"rtv03_baseball_03\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11442.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11442\", \"source_path\": \"data/raw/mini_cure_or/test/11442.jpg\", \"source_selection_id\": \"rtv03_baseball_03\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11442.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11442\", \"source_path\": \"data/raw/mini_cure_or/test/11442.jpg\", \"source_selection_id\": \"rtv03_baseball_03\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11442.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11442\", \"source_path\": \"data/raw/mini_cure_or/test/11442.jpg\", \"source_selection_id\": \"rtv03_baseball_03\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11442.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11442\", \"source_path\": \"data/raw/mini_cure_or/test/11442.jpg\", \"source_selection_id\": \"rtv03_baseball_03\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11677.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11677\", \"source_path\": \"data/raw/mini_cure_or/test/11677.jpg\", \"source_selection_id\": \"rtv03_baseball_04\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11677.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11677\", \"source_path\": \"data/raw/mini_cure_or/test/11677.jpg\", \"source_selection_id\": \"rtv03_baseball_04\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11677.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11677\", \"source_path\": \"data/raw/mini_cure_or/test/11677.jpg\", \"source_selection_id\": \"rtv03_baseball_04\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11677.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11677\", \"source_path\": \"data/raw/mini_cure_or/test/11677.jpg\", \"source_selection_id\": \"rtv03_baseball_04\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11677.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11677\", \"source_path\": \"data/raw/mini_cure_or/test/11677.jpg\", \"source_selection_id\": \"rtv03_baseball_04\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11677.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11677\", \"source_path\": \"data/raw/mini_cure_or/test/11677.jpg\", \"source_selection_id\": \"rtv03_baseball_04\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11677.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11677\", \"source_path\": \"data/raw/mini_cure_or/test/11677.jpg\", \"source_selection_id\": \"rtv03_baseball_04\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11677.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11677\", \"source_path\": \"data/raw/mini_cure_or/test/11677.jpg\", \"source_selection_id\": \"rtv03_baseball_04\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13590.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13590\", \"source_path\": \"data/raw/mini_cure_or/test/13590.jpg\", \"source_selection_id\": \"rtv03_baseball_05\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13590.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13590\", \"source_path\": \"data/raw/mini_cure_or/test/13590.jpg\", \"source_selection_id\": \"rtv03_baseball_05\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13590.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13590\", \"source_path\": \"data/raw/mini_cure_or/test/13590.jpg\", \"source_selection_id\": \"rtv03_baseball_05\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13590.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13590\", \"source_path\": \"data/raw/mini_cure_or/test/13590.jpg\", \"source_selection_id\": \"rtv03_baseball_05\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13590.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13590\", \"source_path\": \"data/raw/mini_cure_or/test/13590.jpg\", \"source_selection_id\": \"rtv03_baseball_05\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13590.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13590\", \"source_path\": \"data/raw/mini_cure_or/test/13590.jpg\", \"source_selection_id\": \"rtv03_baseball_05\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13590.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13590\", \"source_path\": \"data/raw/mini_cure_or/test/13590.jpg\", \"source_selection_id\": \"rtv03_baseball_05\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13590.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13590\", \"source_path\": \"data/raw/mini_cure_or/test/13590.jpg\", \"source_selection_id\": \"rtv03_baseball_05\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13724.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13724\", \"source_path\": \"data/raw/mini_cure_or/test/13724.jpg\", \"source_selection_id\": \"rtv03_baseball_06\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13724.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13724\", \"source_path\": \"data/raw/mini_cure_or/test/13724.jpg\", \"source_selection_id\": \"rtv03_baseball_06\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13724.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13724\", \"source_path\": \"data/raw/mini_cure_or/test/13724.jpg\", \"source_selection_id\": \"rtv03_baseball_06\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13724.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13724\", \"source_path\": \"data/raw/mini_cure_or/test/13724.jpg\", \"source_selection_id\": \"rtv03_baseball_06\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13724.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13724\", \"source_path\": \"data/raw/mini_cure_or/test/13724.jpg\", \"source_selection_id\": \"rtv03_baseball_06\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13724.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13724\", \"source_path\": \"data/raw/mini_cure_or/test/13724.jpg\", \"source_selection_id\": \"rtv03_baseball_06\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13724.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13724\", \"source_path\": \"data/raw/mini_cure_or/test/13724.jpg\", \"source_selection_id\": \"rtv03_baseball_06\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13724.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13724\", \"source_path\": \"data/raw/mini_cure_or/test/13724.jpg\", \"source_selection_id\": \"rtv03_baseball_06\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/14773.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__14773\", \"source_path\": \"data/raw/mini_cure_or/test/14773.jpg\", \"source_selection_id\": \"rtv03_baseball_07\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/14773.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__14773\", \"source_path\": \"data/raw/mini_cure_or/test/14773.jpg\", \"source_selection_id\": \"rtv03_baseball_07\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/14773.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__14773\", \"source_path\": \"data/raw/mini_cure_or/test/14773.jpg\", \"source_selection_id\": \"rtv03_baseball_07\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/14773.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__14773\", \"source_path\": \"data/raw/mini_cure_or/test/14773.jpg\", \"source_selection_id\": \"rtv03_baseball_07\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/14773.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__14773\", \"source_path\": \"data/raw/mini_cure_or/test/14773.jpg\", \"source_selection_id\": \"rtv03_baseball_07\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/14773.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__14773\", \"source_path\": \"data/raw/mini_cure_or/test/14773.jpg\", \"source_selection_id\": \"rtv03_baseball_07\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/14773.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__14773\", \"source_path\": \"data/raw/mini_cure_or/test/14773.jpg\", \"source_selection_id\": \"rtv03_baseball_07\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/14773.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__14773\", \"source_path\": \"data/raw/mini_cure_or/test/14773.jpg\", \"source_selection_id\": \"rtv03_baseball_07\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15024.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15024\", \"source_path\": \"data/raw/mini_cure_or/test/15024.jpg\", \"source_selection_id\": \"rtv03_baseball_08\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15024.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15024\", \"source_path\": \"data/raw/mini_cure_or/test/15024.jpg\", \"source_selection_id\": \"rtv03_baseball_08\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15024.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15024\", \"source_path\": \"data/raw/mini_cure_or/test/15024.jpg\", \"source_selection_id\": \"rtv03_baseball_08\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15024.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15024\", \"source_path\": \"data/raw/mini_cure_or/test/15024.jpg\", \"source_selection_id\": \"rtv03_baseball_08\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15024.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15024\", \"source_path\": \"data/raw/mini_cure_or/test/15024.jpg\", \"source_selection_id\": \"rtv03_baseball_08\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15024.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15024\", \"source_path\": \"data/raw/mini_cure_or/test/15024.jpg\", \"source_selection_id\": \"rtv03_baseball_08\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15024.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15024\", \"source_path\": \"data/raw/mini_cure_or/test/15024.jpg\", \"source_selection_id\": \"rtv03_baseball_08\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15024.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15024\", \"source_path\": \"data/raw/mini_cure_or/test/15024.jpg\", \"source_selection_id\": \"rtv03_baseball_08\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15309.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15309\", \"source_path\": \"data/raw/mini_cure_or/test/15309.jpg\", \"source_selection_id\": \"rtv03_baseball_09\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15309.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15309\", \"source_path\": \"data/raw/mini_cure_or/test/15309.jpg\", \"source_selection_id\": \"rtv03_baseball_09\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15309.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15309\", \"source_path\": \"data/raw/mini_cure_or/test/15309.jpg\", \"source_selection_id\": \"rtv03_baseball_09\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15309.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15309\", \"source_path\": \"data/raw/mini_cure_or/test/15309.jpg\", \"source_selection_id\": \"rtv03_baseball_09\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15309.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15309\", \"source_path\": \"data/raw/mini_cure_or/test/15309.jpg\", \"source_selection_id\": \"rtv03_baseball_09\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15309.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15309\", \"source_path\": \"data/raw/mini_cure_or/test/15309.jpg\", \"source_selection_id\": \"rtv03_baseball_09\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15309.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15309\", \"source_path\": \"data/raw/mini_cure_or/test/15309.jpg\", \"source_selection_id\": \"rtv03_baseball_09\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15309.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15309\", \"source_path\": \"data/raw/mini_cure_or/test/15309.jpg\", \"source_selection_id\": \"rtv03_baseball_09\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15621.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15621\", \"source_path\": \"data/raw/mini_cure_or/test/15621.jpg\", \"source_selection_id\": \"rtv03_baseball_10\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15621.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15621\", \"source_path\": \"data/raw/mini_cure_or/test/15621.jpg\", \"source_selection_id\": \"rtv03_baseball_10\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15621.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15621\", \"source_path\": \"data/raw/mini_cure_or/test/15621.jpg\", \"source_selection_id\": \"rtv03_baseball_10\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15621.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15621\", \"source_path\": \"data/raw/mini_cure_or/test/15621.jpg\", \"source_selection_id\": \"rtv03_baseball_10\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15621.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15621\", \"source_path\": \"data/raw/mini_cure_or/test/15621.jpg\", \"source_selection_id\": \"rtv03_baseball_10\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15621.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15621\", \"source_path\": \"data/raw/mini_cure_or/test/15621.jpg\", \"source_selection_id\": \"rtv03_baseball_10\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15621.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15621\", \"source_path\": \"data/raw/mini_cure_or/test/15621.jpg\", \"source_selection_id\": \"rtv03_baseball_10\"}\n{\"answer_display\": \"baseball\", \"answer_letter\": \"A\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15621.jpg\", \"label\": \"baseball\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15621\", \"source_path\": \"data/raw/mini_cure_or/test/15621.jpg\", \"source_selection_id\": \"rtv03_baseball_10\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11247.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11247\", \"source_path\": \"data/raw/mini_cure_or/test/11247.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_01\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11247.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11247\", \"source_path\": \"data/raw/mini_cure_or/test/11247.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_01\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11247.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11247\", \"source_path\": \"data/raw/mini_cure_or/test/11247.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_01\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11247.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11247\", \"source_path\": \"data/raw/mini_cure_or/test/11247.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_01\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11247.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11247\", \"source_path\": \"data/raw/mini_cure_or/test/11247.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_01\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11247.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11247\", \"source_path\": \"data/raw/mini_cure_or/test/11247.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_01\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11247.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11247\", \"source_path\": \"data/raw/mini_cure_or/test/11247.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_01\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11247.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11247\", \"source_path\": \"data/raw/mini_cure_or/test/11247.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_01\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/12815.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__12815\", \"source_path\": \"data/raw/mini_cure_or/test/12815.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_02\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/12815.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__12815\", \"source_path\": \"data/raw/mini_cure_or/test/12815.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_02\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/12815.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__12815\", \"source_path\": \"data/raw/mini_cure_or/test/12815.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_02\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/12815.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__12815\", \"source_path\": \"data/raw/mini_cure_or/test/12815.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_02\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/12815.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__12815\", \"source_path\": \"data/raw/mini_cure_or/test/12815.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_02\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/12815.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__12815\", \"source_path\": \"data/raw/mini_cure_or/test/12815.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_02\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/12815.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__12815\", \"source_path\": \"data/raw/mini_cure_or/test/12815.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_02\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/12815.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__12815\", \"source_path\": \"data/raw/mini_cure_or/test/12815.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_02\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13506.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13506\", \"source_path\": \"data/raw/mini_cure_or/test/13506.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_03\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13506.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13506\", \"source_path\": \"data/raw/mini_cure_or/test/13506.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_03\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13506.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13506\", \"source_path\": \"data/raw/mini_cure_or/test/13506.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_03\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13506.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13506\", \"source_path\": \"data/raw/mini_cure_or/test/13506.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_03\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13506.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13506\", \"source_path\": \"data/raw/mini_cure_or/test/13506.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_03\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13506.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13506\", \"source_path\": \"data/raw/mini_cure_or/test/13506.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_03\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13506.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13506\", \"source_path\": \"data/raw/mini_cure_or/test/13506.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_03\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13506.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13506\", \"source_path\": \"data/raw/mini_cure_or/test/13506.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_03\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/14220.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__14220\", \"source_path\": \"data/raw/mini_cure_or/test/14220.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_04\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/14220.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__14220\", \"source_path\": \"data/raw/mini_cure_or/test/14220.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_04\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/14220.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__14220\", \"source_path\": \"data/raw/mini_cure_or/test/14220.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_04\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/14220.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__14220\", \"source_path\": \"data/raw/mini_cure_or/test/14220.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_04\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/14220.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__14220\", \"source_path\": \"data/raw/mini_cure_or/test/14220.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_04\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/14220.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__14220\", \"source_path\": \"data/raw/mini_cure_or/test/14220.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_04\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/14220.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__14220\", \"source_path\": \"data/raw/mini_cure_or/test/14220.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_04\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/14220.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__14220\", \"source_path\": \"data/raw/mini_cure_or/test/14220.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_04\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/14405.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__14405\", \"source_path\": \"data/raw/mini_cure_or/test/14405.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_05\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/14405.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__14405\", \"source_path\": \"data/raw/mini_cure_or/test/14405.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_05\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/14405.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__14405\", \"source_path\": \"data/raw/mini_cure_or/test/14405.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_05\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/14405.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__14405\", \"source_path\": \"data/raw/mini_cure_or/test/14405.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_05\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/14405.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__14405\", \"source_path\": \"data/raw/mini_cure_or/test/14405.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_05\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/14405.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__14405\", \"source_path\": \"data/raw/mini_cure_or/test/14405.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_05\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/14405.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__14405\", \"source_path\": \"data/raw/mini_cure_or/test/14405.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_05\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/14405.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__14405\", \"source_path\": \"data/raw/mini_cure_or/test/14405.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_05\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/14978.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__14978\", \"source_path\": \"data/raw/mini_cure_or/test/14978.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_06\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/14978.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__14978\", \"source_path\": \"data/raw/mini_cure_or/test/14978.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_06\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/14978.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__14978\", \"source_path\": \"data/raw/mini_cure_or/test/14978.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_06\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/14978.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__14978\", \"source_path\": \"data/raw/mini_cure_or/test/14978.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_06\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/14978.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__14978\", \"source_path\": \"data/raw/mini_cure_or/test/14978.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_06\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/14978.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__14978\", \"source_path\": \"data/raw/mini_cure_or/test/14978.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_06\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/14978.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__14978\", \"source_path\": \"data/raw/mini_cure_or/test/14978.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_06\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/14978.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__14978\", \"source_path\": \"data/raw/mini_cure_or/test/14978.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_06\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15012.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15012\", \"source_path\": \"data/raw/mini_cure_or/test/15012.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_07\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15012.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15012\", \"source_path\": \"data/raw/mini_cure_or/test/15012.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_07\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15012.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15012\", \"source_path\": \"data/raw/mini_cure_or/test/15012.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_07\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15012.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15012\", \"source_path\": \"data/raw/mini_cure_or/test/15012.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_07\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15012.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15012\", \"source_path\": \"data/raw/mini_cure_or/test/15012.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_07\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15012.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15012\", \"source_path\": \"data/raw/mini_cure_or/test/15012.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_07\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15012.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15012\", \"source_path\": \"data/raw/mini_cure_or/test/15012.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_07\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15012.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15012\", \"source_path\": \"data/raw/mini_cure_or/test/15012.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_07\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15207.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15207\", \"source_path\": \"data/raw/mini_cure_or/test/15207.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_08\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15207.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15207\", \"source_path\": \"data/raw/mini_cure_or/test/15207.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_08\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15207.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15207\", \"source_path\": \"data/raw/mini_cure_or/test/15207.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_08\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15207.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15207\", \"source_path\": \"data/raw/mini_cure_or/test/15207.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_08\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15207.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15207\", \"source_path\": \"data/raw/mini_cure_or/test/15207.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_08\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15207.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15207\", \"source_path\": \"data/raw/mini_cure_or/test/15207.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_08\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15207.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15207\", \"source_path\": \"data/raw/mini_cure_or/test/15207.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_08\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15207.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15207\", \"source_path\": \"data/raw/mini_cure_or/test/15207.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_08\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15222.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15222\", \"source_path\": \"data/raw/mini_cure_or/test/15222.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_09\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15222.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15222\", \"source_path\": \"data/raw/mini_cure_or/test/15222.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_09\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15222.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15222\", \"source_path\": \"data/raw/mini_cure_or/test/15222.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_09\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15222.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15222\", \"source_path\": \"data/raw/mini_cure_or/test/15222.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_09\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15222.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15222\", \"source_path\": \"data/raw/mini_cure_or/test/15222.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_09\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15222.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15222\", \"source_path\": \"data/raw/mini_cure_or/test/15222.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_09\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15222.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15222\", \"source_path\": \"data/raw/mini_cure_or/test/15222.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_09\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15222.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15222\", \"source_path\": \"data/raw/mini_cure_or/test/15222.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_09\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15381.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15381\", \"source_path\": \"data/raw/mini_cure_or/test/15381.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_10\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15381.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15381\", \"source_path\": \"data/raw/mini_cure_or/test/15381.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_10\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15381.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15381\", \"source_path\": \"data/raw/mini_cure_or/test/15381.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_10\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15381.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15381\", \"source_path\": \"data/raw/mini_cure_or/test/15381.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_10\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15381.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15381\", \"source_path\": \"data/raw/mini_cure_or/test/15381.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_10\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15381.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15381\", \"source_path\": \"data/raw/mini_cure_or/test/15381.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_10\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15381.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15381\", \"source_path\": \"data/raw/mini_cure_or/test/15381.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_10\"}\n{\"answer_display\": \"calcium bottle\", \"answer_letter\": \"B\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15381.jpg\", \"label\": \"calcium_bottle\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15381\", \"source_path\": \"data/raw/mini_cure_or/test/15381.jpg\", \"source_selection_id\": \"rtv03_calcium_bottle_10\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/10243.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__10243\", \"source_path\": \"data/raw/mini_cure_or/test/10243.jpg\", \"source_selection_id\": \"rtv03_canon_camera_01\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/10243.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__10243\", \"source_path\": \"data/raw/mini_cure_or/test/10243.jpg\", \"source_selection_id\": \"rtv03_canon_camera_01\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/10243.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__10243\", \"source_path\": \"data/raw/mini_cure_or/test/10243.jpg\", \"source_selection_id\": \"rtv03_canon_camera_01\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/10243.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__10243\", \"source_path\": \"data/raw/mini_cure_or/test/10243.jpg\", \"source_selection_id\": \"rtv03_canon_camera_01\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/10243.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__10243\", \"source_path\": \"data/raw/mini_cure_or/test/10243.jpg\", \"source_selection_id\": \"rtv03_canon_camera_01\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/10243.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__10243\", \"source_path\": \"data/raw/mini_cure_or/test/10243.jpg\", \"source_selection_id\": \"rtv03_canon_camera_01\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/10243.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__10243\", \"source_path\": \"data/raw/mini_cure_or/test/10243.jpg\", \"source_selection_id\": \"rtv03_canon_camera_01\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/10243.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__10243\", \"source_path\": \"data/raw/mini_cure_or/test/10243.jpg\", \"source_selection_id\": \"rtv03_canon_camera_01\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/10530.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__10530\", \"source_path\": \"data/raw/mini_cure_or/test/10530.jpg\", \"source_selection_id\": \"rtv03_canon_camera_02\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/10530.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__10530\", \"source_path\": \"data/raw/mini_cure_or/test/10530.jpg\", \"source_selection_id\": \"rtv03_canon_camera_02\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/10530.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__10530\", \"source_path\": \"data/raw/mini_cure_or/test/10530.jpg\", \"source_selection_id\": \"rtv03_canon_camera_02\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/10530.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__10530\", \"source_path\": \"data/raw/mini_cure_or/test/10530.jpg\", \"source_selection_id\": \"rtv03_canon_camera_02\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/10530.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__10530\", \"source_path\": \"data/raw/mini_cure_or/test/10530.jpg\", \"source_selection_id\": \"rtv03_canon_camera_02\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/10530.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__10530\", \"source_path\": \"data/raw/mini_cure_or/test/10530.jpg\", \"source_selection_id\": \"rtv03_canon_camera_02\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/10530.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__10530\", \"source_path\": \"data/raw/mini_cure_or/test/10530.jpg\", \"source_selection_id\": \"rtv03_canon_camera_02\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/10530.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__10530\", \"source_path\": \"data/raw/mini_cure_or/test/10530.jpg\", \"source_selection_id\": \"rtv03_canon_camera_02\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/12528.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__12528\", \"source_path\": \"data/raw/mini_cure_or/test/12528.jpg\", \"source_selection_id\": \"rtv03_canon_camera_03\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/12528.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__12528\", \"source_path\": \"data/raw/mini_cure_or/test/12528.jpg\", \"source_selection_id\": \"rtv03_canon_camera_03\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/12528.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__12528\", \"source_path\": \"data/raw/mini_cure_or/test/12528.jpg\", \"source_selection_id\": \"rtv03_canon_camera_03\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/12528.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__12528\", \"source_path\": \"data/raw/mini_cure_or/test/12528.jpg\", \"source_selection_id\": \"rtv03_canon_camera_03\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/12528.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__12528\", \"source_path\": \"data/raw/mini_cure_or/test/12528.jpg\", \"source_selection_id\": \"rtv03_canon_camera_03\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/12528.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__12528\", \"source_path\": \"data/raw/mini_cure_or/test/12528.jpg\", \"source_selection_id\": \"rtv03_canon_camera_03\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/12528.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__12528\", \"source_path\": \"data/raw/mini_cure_or/test/12528.jpg\", \"source_selection_id\": \"rtv03_canon_camera_03\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/12528.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__12528\", \"source_path\": \"data/raw/mini_cure_or/test/12528.jpg\", \"source_selection_id\": \"rtv03_canon_camera_03\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13124.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13124\", \"source_path\": \"data/raw/mini_cure_or/test/13124.jpg\", \"source_selection_id\": \"rtv03_canon_camera_04\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13124.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13124\", \"source_path\": \"data/raw/mini_cure_or/test/13124.jpg\", \"source_selection_id\": \"rtv03_canon_camera_04\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13124.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13124\", \"source_path\": \"data/raw/mini_cure_or/test/13124.jpg\", \"source_selection_id\": \"rtv03_canon_camera_04\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13124.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13124\", \"source_path\": \"data/raw/mini_cure_or/test/13124.jpg\", \"source_selection_id\": \"rtv03_canon_camera_04\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13124.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13124\", \"source_path\": \"data/raw/mini_cure_or/test/13124.jpg\", \"source_selection_id\": \"rtv03_canon_camera_04\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13124.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13124\", \"source_path\": \"data/raw/mini_cure_or/test/13124.jpg\", \"source_selection_id\": \"rtv03_canon_camera_04\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13124.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13124\", \"source_path\": \"data/raw/mini_cure_or/test/13124.jpg\", \"source_selection_id\": \"rtv03_canon_camera_04\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13124.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13124\", \"source_path\": \"data/raw/mini_cure_or/test/13124.jpg\", \"source_selection_id\": \"rtv03_canon_camera_04\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13641.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13641\", \"source_path\": \"data/raw/mini_cure_or/test/13641.jpg\", \"source_selection_id\": \"rtv03_canon_camera_05\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13641.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13641\", \"source_path\": \"data/raw/mini_cure_or/test/13641.jpg\", \"source_selection_id\": \"rtv03_canon_camera_05\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13641.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13641\", \"source_path\": \"data/raw/mini_cure_or/test/13641.jpg\", \"source_selection_id\": \"rtv03_canon_camera_05\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13641.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13641\", \"source_path\": \"data/raw/mini_cure_or/test/13641.jpg\", \"source_selection_id\": \"rtv03_canon_camera_05\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13641.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13641\", \"source_path\": \"data/raw/mini_cure_or/test/13641.jpg\", \"source_selection_id\": \"rtv03_canon_camera_05\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13641.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13641\", \"source_path\": \"data/raw/mini_cure_or/test/13641.jpg\", \"source_selection_id\": \"rtv03_canon_camera_05\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13641.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13641\", \"source_path\": \"data/raw/mini_cure_or/test/13641.jpg\", \"source_selection_id\": \"rtv03_canon_camera_05\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13641.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13641\", \"source_path\": \"data/raw/mini_cure_or/test/13641.jpg\", \"source_selection_id\": \"rtv03_canon_camera_05\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13924.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13924\", \"source_path\": \"data/raw/mini_cure_or/test/13924.jpg\", \"source_selection_id\": \"rtv03_canon_camera_06\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13924.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13924\", \"source_path\": \"data/raw/mini_cure_or/test/13924.jpg\", \"source_selection_id\": \"rtv03_canon_camera_06\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13924.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13924\", \"source_path\": \"data/raw/mini_cure_or/test/13924.jpg\", \"source_selection_id\": \"rtv03_canon_camera_06\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13924.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13924\", \"source_path\": \"data/raw/mini_cure_or/test/13924.jpg\", \"source_selection_id\": \"rtv03_canon_camera_06\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13924.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13924\", \"source_path\": \"data/raw/mini_cure_or/test/13924.jpg\", \"source_selection_id\": \"rtv03_canon_camera_06\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13924.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13924\", \"source_path\": \"data/raw/mini_cure_or/test/13924.jpg\", \"source_selection_id\": \"rtv03_canon_camera_06\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13924.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13924\", \"source_path\": \"data/raw/mini_cure_or/test/13924.jpg\", \"source_selection_id\": \"rtv03_canon_camera_06\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13924.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13924\", \"source_path\": \"data/raw/mini_cure_or/test/13924.jpg\", \"source_selection_id\": \"rtv03_canon_camera_06\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/14484.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__14484\", \"source_path\": \"data/raw/mini_cure_or/test/14484.jpg\", \"source_selection_id\": \"rtv03_canon_camera_07\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/14484.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__14484\", \"source_path\": \"data/raw/mini_cure_or/test/14484.jpg\", \"source_selection_id\": \"rtv03_canon_camera_07\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/14484.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__14484\", \"source_path\": \"data/raw/mini_cure_or/test/14484.jpg\", \"source_selection_id\": \"rtv03_canon_camera_07\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/14484.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__14484\", \"source_path\": \"data/raw/mini_cure_or/test/14484.jpg\", \"source_selection_id\": \"rtv03_canon_camera_07\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/14484.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__14484\", \"source_path\": \"data/raw/mini_cure_or/test/14484.jpg\", \"source_selection_id\": \"rtv03_canon_camera_07\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/14484.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__14484\", \"source_path\": \"data/raw/mini_cure_or/test/14484.jpg\", \"source_selection_id\": \"rtv03_canon_camera_07\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/14484.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__14484\", \"source_path\": \"data/raw/mini_cure_or/test/14484.jpg\", \"source_selection_id\": \"rtv03_canon_camera_07\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/14484.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__14484\", \"source_path\": \"data/raw/mini_cure_or/test/14484.jpg\", \"source_selection_id\": \"rtv03_canon_camera_07\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15716.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15716\", \"source_path\": \"data/raw/mini_cure_or/test/15716.jpg\", \"source_selection_id\": \"rtv03_canon_camera_08\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15716.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15716\", \"source_path\": \"data/raw/mini_cure_or/test/15716.jpg\", \"source_selection_id\": \"rtv03_canon_camera_08\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15716.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15716\", \"source_path\": \"data/raw/mini_cure_or/test/15716.jpg\", \"source_selection_id\": \"rtv03_canon_camera_08\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15716.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15716\", \"source_path\": \"data/raw/mini_cure_or/test/15716.jpg\", \"source_selection_id\": \"rtv03_canon_camera_08\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15716.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15716\", \"source_path\": \"data/raw/mini_cure_or/test/15716.jpg\", \"source_selection_id\": \"rtv03_canon_camera_08\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15716.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15716\", \"source_path\": \"data/raw/mini_cure_or/test/15716.jpg\", \"source_selection_id\": \"rtv03_canon_camera_08\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15716.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15716\", \"source_path\": \"data/raw/mini_cure_or/test/15716.jpg\", \"source_selection_id\": \"rtv03_canon_camera_08\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15716.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15716\", \"source_path\": \"data/raw/mini_cure_or/test/15716.jpg\", \"source_selection_id\": \"rtv03_canon_camera_08\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/16451.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__16451\", \"source_path\": \"data/raw/mini_cure_or/test/16451.jpg\", \"source_selection_id\": \"rtv03_canon_camera_09\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/16451.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__16451\", \"source_path\": \"data/raw/mini_cure_or/test/16451.jpg\", \"source_selection_id\": \"rtv03_canon_camera_09\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/16451.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__16451\", \"source_path\": \"data/raw/mini_cure_or/test/16451.jpg\", \"source_selection_id\": \"rtv03_canon_camera_09\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/16451.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__16451\", \"source_path\": \"data/raw/mini_cure_or/test/16451.jpg\", \"source_selection_id\": \"rtv03_canon_camera_09\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/16451.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__16451\", \"source_path\": \"data/raw/mini_cure_or/test/16451.jpg\", \"source_selection_id\": \"rtv03_canon_camera_09\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/16451.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__16451\", \"source_path\": \"data/raw/mini_cure_or/test/16451.jpg\", \"source_selection_id\": \"rtv03_canon_camera_09\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/16451.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__16451\", \"source_path\": \"data/raw/mini_cure_or/test/16451.jpg\", \"source_selection_id\": \"rtv03_canon_camera_09\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/16451.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__16451\", \"source_path\": \"data/raw/mini_cure_or/test/16451.jpg\", \"source_selection_id\": \"rtv03_canon_camera_09\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/16481.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__16481\", \"source_path\": \"data/raw/mini_cure_or/test/16481.jpg\", \"source_selection_id\": \"rtv03_canon_camera_10\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/16481.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__16481\", \"source_path\": \"data/raw/mini_cure_or/test/16481.jpg\", \"source_selection_id\": \"rtv03_canon_camera_10\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/16481.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__16481\", \"source_path\": \"data/raw/mini_cure_or/test/16481.jpg\", \"source_selection_id\": \"rtv03_canon_camera_10\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/16481.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__16481\", \"source_path\": \"data/raw/mini_cure_or/test/16481.jpg\", \"source_selection_id\": \"rtv03_canon_camera_10\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/16481.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__16481\", \"source_path\": \"data/raw/mini_cure_or/test/16481.jpg\", \"source_selection_id\": \"rtv03_canon_camera_10\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/16481.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__16481\", \"source_path\": \"data/raw/mini_cure_or/test/16481.jpg\", \"source_selection_id\": \"rtv03_canon_camera_10\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/16481.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__16481\", \"source_path\": \"data/raw/mini_cure_or/test/16481.jpg\", \"source_selection_id\": \"rtv03_canon_camera_10\"}\n{\"answer_display\": \"Canon camera\", \"answer_letter\": \"C\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/16481.jpg\", \"label\": \"canon_camera\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__16481\", \"source_path\": \"data/raw/mini_cure_or/test/16481.jpg\", \"source_selection_id\": \"rtv03_canon_camera_10\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/09958.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__09958\", \"source_path\": \"data/raw/mini_cure_or/test/09958.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_01\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/09958.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__09958\", \"source_path\": \"data/raw/mini_cure_or/test/09958.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_01\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/09958.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__09958\", \"source_path\": \"data/raw/mini_cure_or/test/09958.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_01\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/09958.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__09958\", \"source_path\": \"data/raw/mini_cure_or/test/09958.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_01\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/09958.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__09958\", \"source_path\": \"data/raw/mini_cure_or/test/09958.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_01\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/09958.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__09958\", \"source_path\": \"data/raw/mini_cure_or/test/09958.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_01\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/09958.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__09958\", \"source_path\": \"data/raw/mini_cure_or/test/09958.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_01\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/09958.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__09958\", \"source_path\": \"data/raw/mini_cure_or/test/09958.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_01\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11887.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11887\", \"source_path\": \"data/raw/mini_cure_or/test/11887.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_02\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11887.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11887\", \"source_path\": \"data/raw/mini_cure_or/test/11887.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_02\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11887.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11887\", \"source_path\": \"data/raw/mini_cure_or/test/11887.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_02\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11887.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11887\", \"source_path\": \"data/raw/mini_cure_or/test/11887.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_02\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11887.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11887\", \"source_path\": \"data/raw/mini_cure_or/test/11887.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_02\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11887.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11887\", \"source_path\": \"data/raw/mini_cure_or/test/11887.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_02\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11887.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11887\", \"source_path\": \"data/raw/mini_cure_or/test/11887.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_02\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11887.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11887\", \"source_path\": \"data/raw/mini_cure_or/test/11887.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_02\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/12043.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__12043\", \"source_path\": \"data/raw/mini_cure_or/test/12043.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_03\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/12043.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__12043\", \"source_path\": \"data/raw/mini_cure_or/test/12043.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_03\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/12043.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__12043\", \"source_path\": \"data/raw/mini_cure_or/test/12043.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_03\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/12043.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__12043\", \"source_path\": \"data/raw/mini_cure_or/test/12043.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_03\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/12043.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__12043\", \"source_path\": \"data/raw/mini_cure_or/test/12043.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_03\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/12043.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__12043\", \"source_path\": \"data/raw/mini_cure_or/test/12043.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_03\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/12043.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__12043\", \"source_path\": \"data/raw/mini_cure_or/test/12043.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_03\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/12043.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__12043\", \"source_path\": \"data/raw/mini_cure_or/test/12043.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_03\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/12706.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__12706\", \"source_path\": \"data/raw/mini_cure_or/test/12706.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_04\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/12706.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__12706\", \"source_path\": \"data/raw/mini_cure_or/test/12706.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_04\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/12706.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__12706\", \"source_path\": \"data/raw/mini_cure_or/test/12706.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_04\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/12706.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__12706\", \"source_path\": \"data/raw/mini_cure_or/test/12706.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_04\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/12706.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__12706\", \"source_path\": \"data/raw/mini_cure_or/test/12706.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_04\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/12706.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__12706\", \"source_path\": \"data/raw/mini_cure_or/test/12706.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_04\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/12706.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__12706\", \"source_path\": \"data/raw/mini_cure_or/test/12706.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_04\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/12706.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__12706\", \"source_path\": \"data/raw/mini_cure_or/test/12706.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_04\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13291.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13291\", \"source_path\": \"data/raw/mini_cure_or/test/13291.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_05\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13291.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13291\", \"source_path\": \"data/raw/mini_cure_or/test/13291.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_05\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13291.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13291\", \"source_path\": \"data/raw/mini_cure_or/test/13291.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_05\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13291.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13291\", \"source_path\": \"data/raw/mini_cure_or/test/13291.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_05\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13291.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13291\", \"source_path\": \"data/raw/mini_cure_or/test/13291.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_05\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13291.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13291\", \"source_path\": \"data/raw/mini_cure_or/test/13291.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_05\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13291.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13291\", \"source_path\": \"data/raw/mini_cure_or/test/13291.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_05\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13291.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13291\", \"source_path\": \"data/raw/mini_cure_or/test/13291.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_05\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13725.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13725\", \"source_path\": \"data/raw/mini_cure_or/test/13725.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_06\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13725.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13725\", \"source_path\": \"data/raw/mini_cure_or/test/13725.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_06\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13725.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13725\", \"source_path\": \"data/raw/mini_cure_or/test/13725.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_06\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13725.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13725\", \"source_path\": \"data/raw/mini_cure_or/test/13725.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_06\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13725.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13725\", \"source_path\": \"data/raw/mini_cure_or/test/13725.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_06\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13725.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13725\", \"source_path\": \"data/raw/mini_cure_or/test/13725.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_06\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13725.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13725\", \"source_path\": \"data/raw/mini_cure_or/test/13725.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_06\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13725.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13725\", \"source_path\": \"data/raw/mini_cure_or/test/13725.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_06\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/14754.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__14754\", \"source_path\": \"data/raw/mini_cure_or/test/14754.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_07\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/14754.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__14754\", \"source_path\": \"data/raw/mini_cure_or/test/14754.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_07\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/14754.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__14754\", \"source_path\": \"data/raw/mini_cure_or/test/14754.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_07\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/14754.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__14754\", \"source_path\": \"data/raw/mini_cure_or/test/14754.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_07\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/14754.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__14754\", \"source_path\": \"data/raw/mini_cure_or/test/14754.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_07\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/14754.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__14754\", \"source_path\": \"data/raw/mini_cure_or/test/14754.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_07\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/14754.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__14754\", \"source_path\": \"data/raw/mini_cure_or/test/14754.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_07\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/14754.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__14754\", \"source_path\": \"data/raw/mini_cure_or/test/14754.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_07\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/14836.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__14836\", \"source_path\": \"data/raw/mini_cure_or/test/14836.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_08\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/14836.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__14836\", \"source_path\": \"data/raw/mini_cure_or/test/14836.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_08\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/14836.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__14836\", \"source_path\": \"data/raw/mini_cure_or/test/14836.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_08\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/14836.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__14836\", \"source_path\": \"data/raw/mini_cure_or/test/14836.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_08\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/14836.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__14836\", \"source_path\": \"data/raw/mini_cure_or/test/14836.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_08\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/14836.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__14836\", \"source_path\": \"data/raw/mini_cure_or/test/14836.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_08\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/14836.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__14836\", \"source_path\": \"data/raw/mini_cure_or/test/14836.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_08\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/14836.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__14836\", \"source_path\": \"data/raw/mini_cure_or/test/14836.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_08\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15150.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15150\", \"source_path\": \"data/raw/mini_cure_or/test/15150.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_09\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15150.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15150\", \"source_path\": \"data/raw/mini_cure_or/test/15150.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_09\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15150.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15150\", \"source_path\": \"data/raw/mini_cure_or/test/15150.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_09\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15150.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15150\", \"source_path\": \"data/raw/mini_cure_or/test/15150.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_09\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15150.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15150\", \"source_path\": \"data/raw/mini_cure_or/test/15150.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_09\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15150.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15150\", \"source_path\": \"data/raw/mini_cure_or/test/15150.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_09\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15150.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15150\", \"source_path\": \"data/raw/mini_cure_or/test/15150.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_09\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15150.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15150\", \"source_path\": \"data/raw/mini_cure_or/test/15150.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_09\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15409.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15409\", \"source_path\": \"data/raw/mini_cure_or/test/15409.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_10\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15409.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15409\", \"source_path\": \"data/raw/mini_cure_or/test/15409.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_10\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15409.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15409\", \"source_path\": \"data/raw/mini_cure_or/test/15409.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_10\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15409.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15409\", \"source_path\": \"data/raw/mini_cure_or/test/15409.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_10\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15409.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15409\", \"source_path\": \"data/raw/mini_cure_or/test/15409.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_10\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15409.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15409\", \"source_path\": \"data/raw/mini_cure_or/test/15409.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_10\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15409.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15409\", \"source_path\": \"data/raw/mini_cure_or/test/15409.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_10\"}\n{\"answer_display\": \"DYMO label maker\", \"answer_letter\": \"D\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15409.jpg\", \"label\": \"dymo_label_maker\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15409\", \"source_path\": \"data/raw/mini_cure_or/test/15409.jpg\", \"source_selection_id\": \"rtv03_dymo_label_maker_10\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11335.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11335\", \"source_path\": \"data/raw/mini_cure_or/test/11335.jpg\", \"source_selection_id\": \"rtv03_hair_brush_01\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11335.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11335\", \"source_path\": \"data/raw/mini_cure_or/test/11335.jpg\", \"source_selection_id\": \"rtv03_hair_brush_01\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11335.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11335\", \"source_path\": \"data/raw/mini_cure_or/test/11335.jpg\", \"source_selection_id\": \"rtv03_hair_brush_01\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11335.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11335\", \"source_path\": \"data/raw/mini_cure_or/test/11335.jpg\", \"source_selection_id\": \"rtv03_hair_brush_01\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11335.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11335\", \"source_path\": \"data/raw/mini_cure_or/test/11335.jpg\", \"source_selection_id\": \"rtv03_hair_brush_01\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11335.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11335\", \"source_path\": \"data/raw/mini_cure_or/test/11335.jpg\", \"source_selection_id\": \"rtv03_hair_brush_01\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11335.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11335\", \"source_path\": \"data/raw/mini_cure_or/test/11335.jpg\", \"source_selection_id\": \"rtv03_hair_brush_01\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11335.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11335\", \"source_path\": \"data/raw/mini_cure_or/test/11335.jpg\", \"source_selection_id\": \"rtv03_hair_brush_01\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11890.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11890\", \"source_path\": \"data/raw/mini_cure_or/test/11890.jpg\", \"source_selection_id\": \"rtv03_hair_brush_02\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11890.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11890\", \"source_path\": \"data/raw/mini_cure_or/test/11890.jpg\", \"source_selection_id\": \"rtv03_hair_brush_02\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11890.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11890\", \"source_path\": \"data/raw/mini_cure_or/test/11890.jpg\", \"source_selection_id\": \"rtv03_hair_brush_02\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11890.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11890\", \"source_path\": \"data/raw/mini_cure_or/test/11890.jpg\", \"source_selection_id\": \"rtv03_hair_brush_02\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11890.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11890\", \"source_path\": \"data/raw/mini_cure_or/test/11890.jpg\", \"source_selection_id\": \"rtv03_hair_brush_02\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11890.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11890\", \"source_path\": \"data/raw/mini_cure_or/test/11890.jpg\", \"source_selection_id\": \"rtv03_hair_brush_02\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11890.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11890\", \"source_path\": \"data/raw/mini_cure_or/test/11890.jpg\", \"source_selection_id\": \"rtv03_hair_brush_02\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11890.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11890\", \"source_path\": \"data/raw/mini_cure_or/test/11890.jpg\", \"source_selection_id\": \"rtv03_hair_brush_02\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13315.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13315\", \"source_path\": \"data/raw/mini_cure_or/test/13315.jpg\", \"source_selection_id\": \"rtv03_hair_brush_03\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13315.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13315\", \"source_path\": \"data/raw/mini_cure_or/test/13315.jpg\", \"source_selection_id\": \"rtv03_hair_brush_03\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13315.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13315\", \"source_path\": \"data/raw/mini_cure_or/test/13315.jpg\", \"source_selection_id\": \"rtv03_hair_brush_03\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13315.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13315\", \"source_path\": \"data/raw/mini_cure_or/test/13315.jpg\", \"source_selection_id\": \"rtv03_hair_brush_03\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13315.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13315\", \"source_path\": \"data/raw/mini_cure_or/test/13315.jpg\", \"source_selection_id\": \"rtv03_hair_brush_03\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13315.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13315\", \"source_path\": \"data/raw/mini_cure_or/test/13315.jpg\", \"source_selection_id\": \"rtv03_hair_brush_03\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13315.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13315\", \"source_path\": \"data/raw/mini_cure_or/test/13315.jpg\", \"source_selection_id\": \"rtv03_hair_brush_03\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13315.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13315\", \"source_path\": \"data/raw/mini_cure_or/test/13315.jpg\", \"source_selection_id\": \"rtv03_hair_brush_03\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13712.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13712\", \"source_path\": \"data/raw/mini_cure_or/test/13712.jpg\", \"source_selection_id\": \"rtv03_hair_brush_04\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13712.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13712\", \"source_path\": \"data/raw/mini_cure_or/test/13712.jpg\", \"source_selection_id\": \"rtv03_hair_brush_04\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13712.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13712\", \"source_path\": \"data/raw/mini_cure_or/test/13712.jpg\", \"source_selection_id\": \"rtv03_hair_brush_04\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13712.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13712\", \"source_path\": \"data/raw/mini_cure_or/test/13712.jpg\", \"source_selection_id\": \"rtv03_hair_brush_04\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13712.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13712\", \"source_path\": \"data/raw/mini_cure_or/test/13712.jpg\", \"source_selection_id\": \"rtv03_hair_brush_04\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13712.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13712\", \"source_path\": \"data/raw/mini_cure_or/test/13712.jpg\", \"source_selection_id\": \"rtv03_hair_brush_04\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13712.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13712\", \"source_path\": \"data/raw/mini_cure_or/test/13712.jpg\", \"source_selection_id\": \"rtv03_hair_brush_04\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13712.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13712\", \"source_path\": \"data/raw/mini_cure_or/test/13712.jpg\", \"source_selection_id\": \"rtv03_hair_brush_04\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13820.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13820\", \"source_path\": \"data/raw/mini_cure_or/test/13820.jpg\", \"source_selection_id\": \"rtv03_hair_brush_05\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13820.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13820\", \"source_path\": \"data/raw/mini_cure_or/test/13820.jpg\", \"source_selection_id\": \"rtv03_hair_brush_05\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13820.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13820\", \"source_path\": \"data/raw/mini_cure_or/test/13820.jpg\", \"source_selection_id\": \"rtv03_hair_brush_05\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13820.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13820\", \"source_path\": \"data/raw/mini_cure_or/test/13820.jpg\", \"source_selection_id\": \"rtv03_hair_brush_05\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13820.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13820\", \"source_path\": \"data/raw/mini_cure_or/test/13820.jpg\", \"source_selection_id\": \"rtv03_hair_brush_05\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13820.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13820\", \"source_path\": \"data/raw/mini_cure_or/test/13820.jpg\", \"source_selection_id\": \"rtv03_hair_brush_05\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13820.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13820\", \"source_path\": \"data/raw/mini_cure_or/test/13820.jpg\", \"source_selection_id\": \"rtv03_hair_brush_05\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13820.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13820\", \"source_path\": \"data/raw/mini_cure_or/test/13820.jpg\", \"source_selection_id\": \"rtv03_hair_brush_05\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15051.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15051\", \"source_path\": \"data/raw/mini_cure_or/test/15051.jpg\", \"source_selection_id\": \"rtv03_hair_brush_06\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15051.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15051\", \"source_path\": \"data/raw/mini_cure_or/test/15051.jpg\", \"source_selection_id\": \"rtv03_hair_brush_06\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15051.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15051\", \"source_path\": \"data/raw/mini_cure_or/test/15051.jpg\", \"source_selection_id\": \"rtv03_hair_brush_06\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15051.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15051\", \"source_path\": \"data/raw/mini_cure_or/test/15051.jpg\", \"source_selection_id\": \"rtv03_hair_brush_06\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15051.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15051\", \"source_path\": \"data/raw/mini_cure_or/test/15051.jpg\", \"source_selection_id\": \"rtv03_hair_brush_06\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15051.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15051\", \"source_path\": \"data/raw/mini_cure_or/test/15051.jpg\", \"source_selection_id\": \"rtv03_hair_brush_06\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15051.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15051\", \"source_path\": \"data/raw/mini_cure_or/test/15051.jpg\", \"source_selection_id\": \"rtv03_hair_brush_06\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15051.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15051\", \"source_path\": \"data/raw/mini_cure_or/test/15051.jpg\", \"source_selection_id\": \"rtv03_hair_brush_06\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15162.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15162\", \"source_path\": \"data/raw/mini_cure_or/test/15162.jpg\", \"source_selection_id\": \"rtv03_hair_brush_07\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15162.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15162\", \"source_path\": \"data/raw/mini_cure_or/test/15162.jpg\", \"source_selection_id\": \"rtv03_hair_brush_07\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15162.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15162\", \"source_path\": \"data/raw/mini_cure_or/test/15162.jpg\", \"source_selection_id\": \"rtv03_hair_brush_07\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15162.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15162\", \"source_path\": \"data/raw/mini_cure_or/test/15162.jpg\", \"source_selection_id\": \"rtv03_hair_brush_07\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15162.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15162\", \"source_path\": \"data/raw/mini_cure_or/test/15162.jpg\", \"source_selection_id\": \"rtv03_hair_brush_07\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15162.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15162\", \"source_path\": \"data/raw/mini_cure_or/test/15162.jpg\", \"source_selection_id\": \"rtv03_hair_brush_07\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15162.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15162\", \"source_path\": \"data/raw/mini_cure_or/test/15162.jpg\", \"source_selection_id\": \"rtv03_hair_brush_07\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15162.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15162\", \"source_path\": \"data/raw/mini_cure_or/test/15162.jpg\", \"source_selection_id\": \"rtv03_hair_brush_07\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15491.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15491\", \"source_path\": \"data/raw/mini_cure_or/test/15491.jpg\", \"source_selection_id\": \"rtv03_hair_brush_08\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15491.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15491\", \"source_path\": \"data/raw/mini_cure_or/test/15491.jpg\", \"source_selection_id\": \"rtv03_hair_brush_08\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15491.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15491\", \"source_path\": \"data/raw/mini_cure_or/test/15491.jpg\", \"source_selection_id\": \"rtv03_hair_brush_08\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15491.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15491\", \"source_path\": \"data/raw/mini_cure_or/test/15491.jpg\", \"source_selection_id\": \"rtv03_hair_brush_08\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15491.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15491\", \"source_path\": \"data/raw/mini_cure_or/test/15491.jpg\", \"source_selection_id\": \"rtv03_hair_brush_08\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15491.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15491\", \"source_path\": \"data/raw/mini_cure_or/test/15491.jpg\", \"source_selection_id\": \"rtv03_hair_brush_08\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15491.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15491\", \"source_path\": \"data/raw/mini_cure_or/test/15491.jpg\", \"source_selection_id\": \"rtv03_hair_brush_08\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15491.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15491\", \"source_path\": \"data/raw/mini_cure_or/test/15491.jpg\", \"source_selection_id\": \"rtv03_hair_brush_08\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15550.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15550\", \"source_path\": \"data/raw/mini_cure_or/test/15550.jpg\", \"source_selection_id\": \"rtv03_hair_brush_09\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15550.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15550\", \"source_path\": \"data/raw/mini_cure_or/test/15550.jpg\", \"source_selection_id\": \"rtv03_hair_brush_09\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15550.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15550\", \"source_path\": \"data/raw/mini_cure_or/test/15550.jpg\", \"source_selection_id\": \"rtv03_hair_brush_09\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15550.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15550\", \"source_path\": \"data/raw/mini_cure_or/test/15550.jpg\", \"source_selection_id\": \"rtv03_hair_brush_09\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15550.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15550\", \"source_path\": \"data/raw/mini_cure_or/test/15550.jpg\", \"source_selection_id\": \"rtv03_hair_brush_09\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15550.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15550\", \"source_path\": \"data/raw/mini_cure_or/test/15550.jpg\", \"source_selection_id\": \"rtv03_hair_brush_09\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15550.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15550\", \"source_path\": \"data/raw/mini_cure_or/test/15550.jpg\", \"source_selection_id\": \"rtv03_hair_brush_09\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15550.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15550\", \"source_path\": \"data/raw/mini_cure_or/test/15550.jpg\", \"source_selection_id\": \"rtv03_hair_brush_09\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15874.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15874\", \"source_path\": \"data/raw/mini_cure_or/test/15874.jpg\", \"source_selection_id\": \"rtv03_hair_brush_10\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15874.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15874\", \"source_path\": \"data/raw/mini_cure_or/test/15874.jpg\", \"source_selection_id\": \"rtv03_hair_brush_10\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15874.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15874\", \"source_path\": \"data/raw/mini_cure_or/test/15874.jpg\", \"source_selection_id\": \"rtv03_hair_brush_10\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15874.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15874\", \"source_path\": \"data/raw/mini_cure_or/test/15874.jpg\", \"source_selection_id\": \"rtv03_hair_brush_10\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15874.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15874\", \"source_path\": \"data/raw/mini_cure_or/test/15874.jpg\", \"source_selection_id\": \"rtv03_hair_brush_10\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15874.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15874\", \"source_path\": \"data/raw/mini_cure_or/test/15874.jpg\", \"source_selection_id\": \"rtv03_hair_brush_10\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15874.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15874\", \"source_path\": \"data/raw/mini_cure_or/test/15874.jpg\", \"source_selection_id\": \"rtv03_hair_brush_10\"}\n{\"answer_display\": \"hair brush\", \"answer_letter\": \"E\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15874.jpg\", \"label\": \"hair_brush\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15874\", \"source_path\": \"data/raw/mini_cure_or/test/15874.jpg\", \"source_selection_id\": \"rtv03_hair_brush_10\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/09904.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__09904\", \"source_path\": \"data/raw/mini_cure_or/test/09904.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_01\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/09904.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__09904\", \"source_path\": \"data/raw/mini_cure_or/test/09904.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_01\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/09904.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__09904\", \"source_path\": \"data/raw/mini_cure_or/test/09904.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_01\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/09904.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__09904\", \"source_path\": \"data/raw/mini_cure_or/test/09904.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_01\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/09904.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__09904\", \"source_path\": \"data/raw/mini_cure_or/test/09904.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_01\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/09904.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__09904\", \"source_path\": \"data/raw/mini_cure_or/test/09904.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_01\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/09904.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__09904\", \"source_path\": \"data/raw/mini_cure_or/test/09904.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_01\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/09904.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__09904\", \"source_path\": \"data/raw/mini_cure_or/test/09904.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_01\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11101.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11101\", \"source_path\": \"data/raw/mini_cure_or/test/11101.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_02\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11101.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11101\", \"source_path\": \"data/raw/mini_cure_or/test/11101.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_02\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11101.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11101\", \"source_path\": \"data/raw/mini_cure_or/test/11101.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_02\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11101.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11101\", \"source_path\": \"data/raw/mini_cure_or/test/11101.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_02\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11101.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11101\", \"source_path\": \"data/raw/mini_cure_or/test/11101.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_02\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11101.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11101\", \"source_path\": \"data/raw/mini_cure_or/test/11101.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_02\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11101.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11101\", \"source_path\": \"data/raw/mini_cure_or/test/11101.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_02\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11101.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11101\", \"source_path\": \"data/raw/mini_cure_or/test/11101.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_02\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11291.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11291\", \"source_path\": \"data/raw/mini_cure_or/test/11291.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_03\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11291.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11291\", \"source_path\": \"data/raw/mini_cure_or/test/11291.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_03\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11291.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11291\", \"source_path\": \"data/raw/mini_cure_or/test/11291.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_03\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11291.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11291\", \"source_path\": \"data/raw/mini_cure_or/test/11291.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_03\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11291.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11291\", \"source_path\": \"data/raw/mini_cure_or/test/11291.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_03\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11291.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11291\", \"source_path\": \"data/raw/mini_cure_or/test/11291.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_03\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11291.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11291\", \"source_path\": \"data/raw/mini_cure_or/test/11291.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_03\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11291.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11291\", \"source_path\": \"data/raw/mini_cure_or/test/11291.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_03\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11758.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11758\", \"source_path\": \"data/raw/mini_cure_or/test/11758.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_04\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11758.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11758\", \"source_path\": \"data/raw/mini_cure_or/test/11758.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_04\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11758.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11758\", \"source_path\": \"data/raw/mini_cure_or/test/11758.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_04\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11758.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11758\", \"source_path\": \"data/raw/mini_cure_or/test/11758.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_04\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11758.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11758\", \"source_path\": \"data/raw/mini_cure_or/test/11758.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_04\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11758.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11758\", \"source_path\": \"data/raw/mini_cure_or/test/11758.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_04\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11758.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11758\", \"source_path\": \"data/raw/mini_cure_or/test/11758.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_04\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11758.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11758\", \"source_path\": \"data/raw/mini_cure_or/test/11758.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_04\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/12550.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__12550\", \"source_path\": \"data/raw/mini_cure_or/test/12550.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_05\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/12550.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__12550\", \"source_path\": \"data/raw/mini_cure_or/test/12550.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_05\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/12550.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__12550\", \"source_path\": \"data/raw/mini_cure_or/test/12550.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_05\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/12550.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__12550\", \"source_path\": \"data/raw/mini_cure_or/test/12550.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_05\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/12550.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__12550\", \"source_path\": \"data/raw/mini_cure_or/test/12550.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_05\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/12550.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__12550\", \"source_path\": \"data/raw/mini_cure_or/test/12550.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_05\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/12550.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__12550\", \"source_path\": \"data/raw/mini_cure_or/test/12550.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_05\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/12550.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__12550\", \"source_path\": \"data/raw/mini_cure_or/test/12550.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_05\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/12704.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__12704\", \"source_path\": \"data/raw/mini_cure_or/test/12704.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_06\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/12704.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__12704\", \"source_path\": \"data/raw/mini_cure_or/test/12704.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_06\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/12704.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__12704\", \"source_path\": \"data/raw/mini_cure_or/test/12704.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_06\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/12704.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__12704\", \"source_path\": \"data/raw/mini_cure_or/test/12704.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_06\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/12704.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__12704\", \"source_path\": \"data/raw/mini_cure_or/test/12704.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_06\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/12704.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__12704\", \"source_path\": \"data/raw/mini_cure_or/test/12704.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_06\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/12704.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__12704\", \"source_path\": \"data/raw/mini_cure_or/test/12704.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_06\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/12704.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__12704\", \"source_path\": \"data/raw/mini_cure_or/test/12704.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_06\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13780.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13780\", \"source_path\": \"data/raw/mini_cure_or/test/13780.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_07\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13780.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13780\", \"source_path\": \"data/raw/mini_cure_or/test/13780.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_07\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13780.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13780\", \"source_path\": \"data/raw/mini_cure_or/test/13780.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_07\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13780.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13780\", \"source_path\": \"data/raw/mini_cure_or/test/13780.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_07\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13780.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13780\", \"source_path\": \"data/raw/mini_cure_or/test/13780.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_07\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13780.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13780\", \"source_path\": \"data/raw/mini_cure_or/test/13780.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_07\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13780.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13780\", \"source_path\": \"data/raw/mini_cure_or/test/13780.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_07\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13780.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13780\", \"source_path\": \"data/raw/mini_cure_or/test/13780.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_07\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13870.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13870\", \"source_path\": \"data/raw/mini_cure_or/test/13870.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_08\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13870.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13870\", \"source_path\": \"data/raw/mini_cure_or/test/13870.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_08\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13870.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13870\", \"source_path\": \"data/raw/mini_cure_or/test/13870.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_08\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13870.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13870\", \"source_path\": \"data/raw/mini_cure_or/test/13870.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_08\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13870.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13870\", \"source_path\": \"data/raw/mini_cure_or/test/13870.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_08\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13870.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13870\", \"source_path\": \"data/raw/mini_cure_or/test/13870.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_08\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13870.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13870\", \"source_path\": \"data/raw/mini_cure_or/test/13870.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_08\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13870.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13870\", \"source_path\": \"data/raw/mini_cure_or/test/13870.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_08\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13897.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13897\", \"source_path\": \"data/raw/mini_cure_or/test/13897.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_09\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13897.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13897\", \"source_path\": \"data/raw/mini_cure_or/test/13897.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_09\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13897.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13897\", \"source_path\": \"data/raw/mini_cure_or/test/13897.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_09\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13897.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13897\", \"source_path\": \"data/raw/mini_cure_or/test/13897.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_09\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13897.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13897\", \"source_path\": \"data/raw/mini_cure_or/test/13897.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_09\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13897.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13897\", \"source_path\": \"data/raw/mini_cure_or/test/13897.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_09\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13897.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13897\", \"source_path\": \"data/raw/mini_cure_or/test/13897.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_09\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13897.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13897\", \"source_path\": \"data/raw/mini_cure_or/test/13897.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_09\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15164.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15164\", \"source_path\": \"data/raw/mini_cure_or/test/15164.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_10\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15164.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15164\", \"source_path\": \"data/raw/mini_cure_or/test/15164.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_10\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15164.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15164\", \"source_path\": \"data/raw/mini_cure_or/test/15164.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_10\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15164.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15164\", \"source_path\": \"data/raw/mini_cure_or/test/15164.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_10\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15164.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15164\", \"source_path\": \"data/raw/mini_cure_or/test/15164.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_10\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15164.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15164\", \"source_path\": \"data/raw/mini_cure_or/test/15164.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_10\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15164.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15164\", \"source_path\": \"data/raw/mini_cure_or/test/15164.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_10\"}\n{\"answer_display\": \"LG cell phone\", \"answer_letter\": \"F\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15164.jpg\", \"label\": \"lg_cell_phone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15164\", \"source_path\": \"data/raw/mini_cure_or/test/15164.jpg\", \"source_selection_id\": \"rtv03_lg_cell_phone_10\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/10714.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__10714\", \"source_path\": \"data/raw/mini_cure_or/test/10714.jpg\", \"source_selection_id\": \"rtv03_pan_01\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/10714.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__10714\", \"source_path\": \"data/raw/mini_cure_or/test/10714.jpg\", \"source_selection_id\": \"rtv03_pan_01\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/10714.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__10714\", \"source_path\": \"data/raw/mini_cure_or/test/10714.jpg\", \"source_selection_id\": \"rtv03_pan_01\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/10714.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__10714\", \"source_path\": \"data/raw/mini_cure_or/test/10714.jpg\", \"source_selection_id\": \"rtv03_pan_01\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/10714.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__10714\", \"source_path\": \"data/raw/mini_cure_or/test/10714.jpg\", \"source_selection_id\": \"rtv03_pan_01\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/10714.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__10714\", \"source_path\": \"data/raw/mini_cure_or/test/10714.jpg\", \"source_selection_id\": \"rtv03_pan_01\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/10714.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__10714\", \"source_path\": \"data/raw/mini_cure_or/test/10714.jpg\", \"source_selection_id\": \"rtv03_pan_01\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/10714.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__10714\", \"source_path\": \"data/raw/mini_cure_or/test/10714.jpg\", \"source_selection_id\": \"rtv03_pan_01\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11285.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11285\", \"source_path\": \"data/raw/mini_cure_or/test/11285.jpg\", \"source_selection_id\": \"rtv03_pan_02\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11285.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11285\", \"source_path\": \"data/raw/mini_cure_or/test/11285.jpg\", \"source_selection_id\": \"rtv03_pan_02\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11285.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11285\", \"source_path\": \"data/raw/mini_cure_or/test/11285.jpg\", \"source_selection_id\": \"rtv03_pan_02\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11285.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11285\", \"source_path\": \"data/raw/mini_cure_or/test/11285.jpg\", \"source_selection_id\": \"rtv03_pan_02\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11285.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11285\", \"source_path\": \"data/raw/mini_cure_or/test/11285.jpg\", \"source_selection_id\": \"rtv03_pan_02\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11285.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11285\", \"source_path\": \"data/raw/mini_cure_or/test/11285.jpg\", \"source_selection_id\": \"rtv03_pan_02\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11285.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11285\", \"source_path\": \"data/raw/mini_cure_or/test/11285.jpg\", \"source_selection_id\": \"rtv03_pan_02\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11285.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11285\", \"source_path\": \"data/raw/mini_cure_or/test/11285.jpg\", \"source_selection_id\": \"rtv03_pan_02\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11460.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11460\", \"source_path\": \"data/raw/mini_cure_or/test/11460.jpg\", \"source_selection_id\": \"rtv03_pan_03\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11460.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11460\", \"source_path\": \"data/raw/mini_cure_or/test/11460.jpg\", \"source_selection_id\": \"rtv03_pan_03\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11460.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11460\", \"source_path\": \"data/raw/mini_cure_or/test/11460.jpg\", \"source_selection_id\": \"rtv03_pan_03\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11460.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11460\", \"source_path\": \"data/raw/mini_cure_or/test/11460.jpg\", \"source_selection_id\": \"rtv03_pan_03\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11460.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11460\", \"source_path\": \"data/raw/mini_cure_or/test/11460.jpg\", \"source_selection_id\": \"rtv03_pan_03\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11460.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11460\", \"source_path\": \"data/raw/mini_cure_or/test/11460.jpg\", \"source_selection_id\": \"rtv03_pan_03\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11460.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11460\", \"source_path\": \"data/raw/mini_cure_or/test/11460.jpg\", \"source_selection_id\": \"rtv03_pan_03\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11460.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11460\", \"source_path\": \"data/raw/mini_cure_or/test/11460.jpg\", \"source_selection_id\": \"rtv03_pan_03\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11608.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11608\", \"source_path\": \"data/raw/mini_cure_or/test/11608.jpg\", \"source_selection_id\": \"rtv03_pan_04\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11608.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11608\", \"source_path\": \"data/raw/mini_cure_or/test/11608.jpg\", \"source_selection_id\": \"rtv03_pan_04\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11608.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11608\", \"source_path\": \"data/raw/mini_cure_or/test/11608.jpg\", \"source_selection_id\": \"rtv03_pan_04\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11608.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11608\", \"source_path\": \"data/raw/mini_cure_or/test/11608.jpg\", \"source_selection_id\": \"rtv03_pan_04\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11608.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11608\", \"source_path\": \"data/raw/mini_cure_or/test/11608.jpg\", \"source_selection_id\": \"rtv03_pan_04\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11608.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11608\", \"source_path\": \"data/raw/mini_cure_or/test/11608.jpg\", \"source_selection_id\": \"rtv03_pan_04\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11608.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11608\", \"source_path\": \"data/raw/mini_cure_or/test/11608.jpg\", \"source_selection_id\": \"rtv03_pan_04\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11608.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11608\", \"source_path\": \"data/raw/mini_cure_or/test/11608.jpg\", \"source_selection_id\": \"rtv03_pan_04\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/12582.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__12582\", \"source_path\": \"data/raw/mini_cure_or/test/12582.jpg\", \"source_selection_id\": \"rtv03_pan_05\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/12582.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__12582\", \"source_path\": \"data/raw/mini_cure_or/test/12582.jpg\", \"source_selection_id\": \"rtv03_pan_05\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/12582.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__12582\", \"source_path\": \"data/raw/mini_cure_or/test/12582.jpg\", \"source_selection_id\": \"rtv03_pan_05\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/12582.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__12582\", \"source_path\": \"data/raw/mini_cure_or/test/12582.jpg\", \"source_selection_id\": \"rtv03_pan_05\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/12582.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__12582\", \"source_path\": \"data/raw/mini_cure_or/test/12582.jpg\", \"source_selection_id\": \"rtv03_pan_05\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/12582.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__12582\", \"source_path\": \"data/raw/mini_cure_or/test/12582.jpg\", \"source_selection_id\": \"rtv03_pan_05\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/12582.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__12582\", \"source_path\": \"data/raw/mini_cure_or/test/12582.jpg\", \"source_selection_id\": \"rtv03_pan_05\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/12582.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__12582\", \"source_path\": \"data/raw/mini_cure_or/test/12582.jpg\", \"source_selection_id\": \"rtv03_pan_05\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/12844.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__12844\", \"source_path\": \"data/raw/mini_cure_or/test/12844.jpg\", \"source_selection_id\": \"rtv03_pan_06\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/12844.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__12844\", \"source_path\": \"data/raw/mini_cure_or/test/12844.jpg\", \"source_selection_id\": \"rtv03_pan_06\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/12844.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__12844\", \"source_path\": \"data/raw/mini_cure_or/test/12844.jpg\", \"source_selection_id\": \"rtv03_pan_06\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/12844.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__12844\", \"source_path\": \"data/raw/mini_cure_or/test/12844.jpg\", \"source_selection_id\": \"rtv03_pan_06\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/12844.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__12844\", \"source_path\": \"data/raw/mini_cure_or/test/12844.jpg\", \"source_selection_id\": \"rtv03_pan_06\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/12844.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__12844\", \"source_path\": \"data/raw/mini_cure_or/test/12844.jpg\", \"source_selection_id\": \"rtv03_pan_06\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/12844.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__12844\", \"source_path\": \"data/raw/mini_cure_or/test/12844.jpg\", \"source_selection_id\": \"rtv03_pan_06\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/12844.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__12844\", \"source_path\": \"data/raw/mini_cure_or/test/12844.jpg\", \"source_selection_id\": \"rtv03_pan_06\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13142.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13142\", \"source_path\": \"data/raw/mini_cure_or/test/13142.jpg\", \"source_selection_id\": \"rtv03_pan_07\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13142.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13142\", \"source_path\": \"data/raw/mini_cure_or/test/13142.jpg\", \"source_selection_id\": \"rtv03_pan_07\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13142.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13142\", \"source_path\": \"data/raw/mini_cure_or/test/13142.jpg\", \"source_selection_id\": \"rtv03_pan_07\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13142.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13142\", \"source_path\": \"data/raw/mini_cure_or/test/13142.jpg\", \"source_selection_id\": \"rtv03_pan_07\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13142.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13142\", \"source_path\": \"data/raw/mini_cure_or/test/13142.jpg\", \"source_selection_id\": \"rtv03_pan_07\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13142.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13142\", \"source_path\": \"data/raw/mini_cure_or/test/13142.jpg\", \"source_selection_id\": \"rtv03_pan_07\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13142.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13142\", \"source_path\": \"data/raw/mini_cure_or/test/13142.jpg\", \"source_selection_id\": \"rtv03_pan_07\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13142.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13142\", \"source_path\": \"data/raw/mini_cure_or/test/13142.jpg\", \"source_selection_id\": \"rtv03_pan_07\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/14382.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__14382\", \"source_path\": \"data/raw/mini_cure_or/test/14382.jpg\", \"source_selection_id\": \"rtv03_pan_08\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/14382.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__14382\", \"source_path\": \"data/raw/mini_cure_or/test/14382.jpg\", \"source_selection_id\": \"rtv03_pan_08\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/14382.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__14382\", \"source_path\": \"data/raw/mini_cure_or/test/14382.jpg\", \"source_selection_id\": \"rtv03_pan_08\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/14382.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__14382\", \"source_path\": \"data/raw/mini_cure_or/test/14382.jpg\", \"source_selection_id\": \"rtv03_pan_08\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/14382.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__14382\", \"source_path\": \"data/raw/mini_cure_or/test/14382.jpg\", \"source_selection_id\": \"rtv03_pan_08\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/14382.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__14382\", \"source_path\": \"data/raw/mini_cure_or/test/14382.jpg\", \"source_selection_id\": \"rtv03_pan_08\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/14382.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__14382\", \"source_path\": \"data/raw/mini_cure_or/test/14382.jpg\", \"source_selection_id\": \"rtv03_pan_08\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/14382.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__14382\", \"source_path\": \"data/raw/mini_cure_or/test/14382.jpg\", \"source_selection_id\": \"rtv03_pan_08\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15772.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15772\", \"source_path\": \"data/raw/mini_cure_or/test/15772.jpg\", \"source_selection_id\": \"rtv03_pan_09\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15772.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15772\", \"source_path\": \"data/raw/mini_cure_or/test/15772.jpg\", \"source_selection_id\": \"rtv03_pan_09\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15772.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15772\", \"source_path\": \"data/raw/mini_cure_or/test/15772.jpg\", \"source_selection_id\": \"rtv03_pan_09\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15772.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15772\", \"source_path\": \"data/raw/mini_cure_or/test/15772.jpg\", \"source_selection_id\": \"rtv03_pan_09\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15772.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15772\", \"source_path\": \"data/raw/mini_cure_or/test/15772.jpg\", \"source_selection_id\": \"rtv03_pan_09\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15772.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15772\", \"source_path\": \"data/raw/mini_cure_or/test/15772.jpg\", \"source_selection_id\": \"rtv03_pan_09\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15772.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15772\", \"source_path\": \"data/raw/mini_cure_or/test/15772.jpg\", \"source_selection_id\": \"rtv03_pan_09\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15772.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15772\", \"source_path\": \"data/raw/mini_cure_or/test/15772.jpg\", \"source_selection_id\": \"rtv03_pan_09\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15808.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15808\", \"source_path\": \"data/raw/mini_cure_or/test/15808.jpg\", \"source_selection_id\": \"rtv03_pan_10\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15808.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15808\", \"source_path\": \"data/raw/mini_cure_or/test/15808.jpg\", \"source_selection_id\": \"rtv03_pan_10\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15808.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15808\", \"source_path\": \"data/raw/mini_cure_or/test/15808.jpg\", \"source_selection_id\": \"rtv03_pan_10\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15808.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15808\", \"source_path\": \"data/raw/mini_cure_or/test/15808.jpg\", \"source_selection_id\": \"rtv03_pan_10\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15808.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15808\", \"source_path\": \"data/raw/mini_cure_or/test/15808.jpg\", \"source_selection_id\": \"rtv03_pan_10\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15808.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15808\", \"source_path\": \"data/raw/mini_cure_or/test/15808.jpg\", \"source_selection_id\": \"rtv03_pan_10\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15808.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15808\", \"source_path\": \"data/raw/mini_cure_or/test/15808.jpg\", \"source_selection_id\": \"rtv03_pan_10\"}\n{\"answer_display\": \"pan\", \"answer_letter\": \"G\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15808.jpg\", \"label\": \"pan\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15808\", \"source_path\": \"data/raw/mini_cure_or/test/15808.jpg\", \"source_selection_id\": \"rtv03_pan_10\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/10640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__10640\", \"source_path\": \"data/raw/mini_cure_or/test/10640.jpg\", \"source_selection_id\": \"rtv03_shoes_01\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/10640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__10640\", \"source_path\": \"data/raw/mini_cure_or/test/10640.jpg\", \"source_selection_id\": \"rtv03_shoes_01\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/10640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__10640\", \"source_path\": \"data/raw/mini_cure_or/test/10640.jpg\", \"source_selection_id\": \"rtv03_shoes_01\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/10640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__10640\", \"source_path\": \"data/raw/mini_cure_or/test/10640.jpg\", \"source_selection_id\": \"rtv03_shoes_01\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/10640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__10640\", \"source_path\": \"data/raw/mini_cure_or/test/10640.jpg\", \"source_selection_id\": \"rtv03_shoes_01\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/10640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__10640\", \"source_path\": \"data/raw/mini_cure_or/test/10640.jpg\", \"source_selection_id\": \"rtv03_shoes_01\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/10640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__10640\", \"source_path\": \"data/raw/mini_cure_or/test/10640.jpg\", \"source_selection_id\": \"rtv03_shoes_01\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/10640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__10640\", \"source_path\": \"data/raw/mini_cure_or/test/10640.jpg\", \"source_selection_id\": \"rtv03_shoes_01\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/10854.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__10854\", \"source_path\": \"data/raw/mini_cure_or/test/10854.jpg\", \"source_selection_id\": \"rtv03_shoes_02\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/10854.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__10854\", \"source_path\": \"data/raw/mini_cure_or/test/10854.jpg\", \"source_selection_id\": \"rtv03_shoes_02\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/10854.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__10854\", \"source_path\": \"data/raw/mini_cure_or/test/10854.jpg\", \"source_selection_id\": \"rtv03_shoes_02\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/10854.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__10854\", \"source_path\": \"data/raw/mini_cure_or/test/10854.jpg\", \"source_selection_id\": \"rtv03_shoes_02\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/10854.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__10854\", \"source_path\": \"data/raw/mini_cure_or/test/10854.jpg\", \"source_selection_id\": \"rtv03_shoes_02\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/10854.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__10854\", \"source_path\": \"data/raw/mini_cure_or/test/10854.jpg\", \"source_selection_id\": \"rtv03_shoes_02\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/10854.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__10854\", \"source_path\": \"data/raw/mini_cure_or/test/10854.jpg\", \"source_selection_id\": \"rtv03_shoes_02\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/10854.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__10854\", \"source_path\": \"data/raw/mini_cure_or/test/10854.jpg\", \"source_selection_id\": \"rtv03_shoes_02\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11446.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11446\", \"source_path\": \"data/raw/mini_cure_or/test/11446.jpg\", \"source_selection_id\": \"rtv03_shoes_03\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11446.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11446\", \"source_path\": \"data/raw/mini_cure_or/test/11446.jpg\", \"source_selection_id\": \"rtv03_shoes_03\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11446.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11446\", \"source_path\": \"data/raw/mini_cure_or/test/11446.jpg\", \"source_selection_id\": \"rtv03_shoes_03\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11446.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11446\", \"source_path\": \"data/raw/mini_cure_or/test/11446.jpg\", \"source_selection_id\": \"rtv03_shoes_03\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11446.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11446\", \"source_path\": \"data/raw/mini_cure_or/test/11446.jpg\", \"source_selection_id\": \"rtv03_shoes_03\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11446.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11446\", \"source_path\": \"data/raw/mini_cure_or/test/11446.jpg\", \"source_selection_id\": \"rtv03_shoes_03\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11446.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11446\", \"source_path\": \"data/raw/mini_cure_or/test/11446.jpg\", \"source_selection_id\": \"rtv03_shoes_03\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11446.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11446\", \"source_path\": \"data/raw/mini_cure_or/test/11446.jpg\", \"source_selection_id\": \"rtv03_shoes_03\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/12605.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__12605\", \"source_path\": \"data/raw/mini_cure_or/test/12605.jpg\", \"source_selection_id\": \"rtv03_shoes_04\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/12605.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__12605\", \"source_path\": \"data/raw/mini_cure_or/test/12605.jpg\", \"source_selection_id\": \"rtv03_shoes_04\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/12605.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__12605\", \"source_path\": \"data/raw/mini_cure_or/test/12605.jpg\", \"source_selection_id\": \"rtv03_shoes_04\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/12605.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__12605\", \"source_path\": \"data/raw/mini_cure_or/test/12605.jpg\", \"source_selection_id\": \"rtv03_shoes_04\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/12605.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__12605\", \"source_path\": \"data/raw/mini_cure_or/test/12605.jpg\", \"source_selection_id\": \"rtv03_shoes_04\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/12605.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__12605\", \"source_path\": \"data/raw/mini_cure_or/test/12605.jpg\", \"source_selection_id\": \"rtv03_shoes_04\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/12605.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__12605\", \"source_path\": \"data/raw/mini_cure_or/test/12605.jpg\", \"source_selection_id\": \"rtv03_shoes_04\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/12605.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__12605\", \"source_path\": \"data/raw/mini_cure_or/test/12605.jpg\", \"source_selection_id\": \"rtv03_shoes_04\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13037.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13037\", \"source_path\": \"data/raw/mini_cure_or/test/13037.jpg\", \"source_selection_id\": \"rtv03_shoes_05\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13037.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13037\", \"source_path\": \"data/raw/mini_cure_or/test/13037.jpg\", \"source_selection_id\": \"rtv03_shoes_05\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13037.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13037\", \"source_path\": \"data/raw/mini_cure_or/test/13037.jpg\", \"source_selection_id\": \"rtv03_shoes_05\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13037.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13037\", \"source_path\": \"data/raw/mini_cure_or/test/13037.jpg\", \"source_selection_id\": \"rtv03_shoes_05\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13037.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13037\", \"source_path\": \"data/raw/mini_cure_or/test/13037.jpg\", \"source_selection_id\": \"rtv03_shoes_05\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13037.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13037\", \"source_path\": \"data/raw/mini_cure_or/test/13037.jpg\", \"source_selection_id\": \"rtv03_shoes_05\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13037.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13037\", \"source_path\": \"data/raw/mini_cure_or/test/13037.jpg\", \"source_selection_id\": \"rtv03_shoes_05\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13037.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13037\", \"source_path\": \"data/raw/mini_cure_or/test/13037.jpg\", \"source_selection_id\": \"rtv03_shoes_05\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13252.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13252\", \"source_path\": \"data/raw/mini_cure_or/test/13252.jpg\", \"source_selection_id\": \"rtv03_shoes_06\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13252.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13252\", \"source_path\": \"data/raw/mini_cure_or/test/13252.jpg\", \"source_selection_id\": \"rtv03_shoes_06\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13252.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13252\", \"source_path\": \"data/raw/mini_cure_or/test/13252.jpg\", \"source_selection_id\": \"rtv03_shoes_06\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13252.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13252\", \"source_path\": \"data/raw/mini_cure_or/test/13252.jpg\", \"source_selection_id\": \"rtv03_shoes_06\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13252.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13252\", \"source_path\": \"data/raw/mini_cure_or/test/13252.jpg\", \"source_selection_id\": \"rtv03_shoes_06\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13252.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13252\", \"source_path\": \"data/raw/mini_cure_or/test/13252.jpg\", \"source_selection_id\": \"rtv03_shoes_06\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13252.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13252\", \"source_path\": \"data/raw/mini_cure_or/test/13252.jpg\", \"source_selection_id\": \"rtv03_shoes_06\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13252.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13252\", \"source_path\": \"data/raw/mini_cure_or/test/13252.jpg\", \"source_selection_id\": \"rtv03_shoes_06\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13819.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13819\", \"source_path\": \"data/raw/mini_cure_or/test/13819.jpg\", \"source_selection_id\": \"rtv03_shoes_07\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13819.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13819\", \"source_path\": \"data/raw/mini_cure_or/test/13819.jpg\", \"source_selection_id\": \"rtv03_shoes_07\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13819.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13819\", \"source_path\": \"data/raw/mini_cure_or/test/13819.jpg\", \"source_selection_id\": \"rtv03_shoes_07\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13819.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13819\", \"source_path\": \"data/raw/mini_cure_or/test/13819.jpg\", \"source_selection_id\": \"rtv03_shoes_07\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13819.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13819\", \"source_path\": \"data/raw/mini_cure_or/test/13819.jpg\", \"source_selection_id\": \"rtv03_shoes_07\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13819.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13819\", \"source_path\": \"data/raw/mini_cure_or/test/13819.jpg\", \"source_selection_id\": \"rtv03_shoes_07\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13819.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13819\", \"source_path\": \"data/raw/mini_cure_or/test/13819.jpg\", \"source_selection_id\": \"rtv03_shoes_07\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13819.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13819\", \"source_path\": \"data/raw/mini_cure_or/test/13819.jpg\", \"source_selection_id\": \"rtv03_shoes_07\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/14426.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__14426\", \"source_path\": \"data/raw/mini_cure_or/test/14426.jpg\", \"source_selection_id\": \"rtv03_shoes_08\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/14426.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__14426\", \"source_path\": \"data/raw/mini_cure_or/test/14426.jpg\", \"source_selection_id\": \"rtv03_shoes_08\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/14426.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__14426\", \"source_path\": \"data/raw/mini_cure_or/test/14426.jpg\", \"source_selection_id\": \"rtv03_shoes_08\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/14426.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__14426\", \"source_path\": \"data/raw/mini_cure_or/test/14426.jpg\", \"source_selection_id\": \"rtv03_shoes_08\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/14426.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__14426\", \"source_path\": \"data/raw/mini_cure_or/test/14426.jpg\", \"source_selection_id\": \"rtv03_shoes_08\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/14426.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__14426\", \"source_path\": \"data/raw/mini_cure_or/test/14426.jpg\", \"source_selection_id\": \"rtv03_shoes_08\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/14426.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__14426\", \"source_path\": \"data/raw/mini_cure_or/test/14426.jpg\", \"source_selection_id\": \"rtv03_shoes_08\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/14426.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__14426\", \"source_path\": \"data/raw/mini_cure_or/test/14426.jpg\", \"source_selection_id\": \"rtv03_shoes_08\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/14640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__14640\", \"source_path\": \"data/raw/mini_cure_or/test/14640.jpg\", \"source_selection_id\": \"rtv03_shoes_09\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/14640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__14640\", \"source_path\": \"data/raw/mini_cure_or/test/14640.jpg\", \"source_selection_id\": \"rtv03_shoes_09\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/14640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__14640\", \"source_path\": \"data/raw/mini_cure_or/test/14640.jpg\", \"source_selection_id\": \"rtv03_shoes_09\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/14640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__14640\", \"source_path\": \"data/raw/mini_cure_or/test/14640.jpg\", \"source_selection_id\": \"rtv03_shoes_09\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/14640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__14640\", \"source_path\": \"data/raw/mini_cure_or/test/14640.jpg\", \"source_selection_id\": \"rtv03_shoes_09\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/14640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__14640\", \"source_path\": \"data/raw/mini_cure_or/test/14640.jpg\", \"source_selection_id\": \"rtv03_shoes_09\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/14640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__14640\", \"source_path\": \"data/raw/mini_cure_or/test/14640.jpg\", \"source_selection_id\": \"rtv03_shoes_09\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/14640.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__14640\", \"source_path\": \"data/raw/mini_cure_or/test/14640.jpg\", \"source_selection_id\": \"rtv03_shoes_09\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15631.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15631\", \"source_path\": \"data/raw/mini_cure_or/test/15631.jpg\", \"source_selection_id\": \"rtv03_shoes_10\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15631.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15631\", \"source_path\": \"data/raw/mini_cure_or/test/15631.jpg\", \"source_selection_id\": \"rtv03_shoes_10\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15631.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15631\", \"source_path\": \"data/raw/mini_cure_or/test/15631.jpg\", \"source_selection_id\": \"rtv03_shoes_10\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15631.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15631\", \"source_path\": \"data/raw/mini_cure_or/test/15631.jpg\", \"source_selection_id\": \"rtv03_shoes_10\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15631.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15631\", \"source_path\": \"data/raw/mini_cure_or/test/15631.jpg\", \"source_selection_id\": \"rtv03_shoes_10\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15631.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15631\", \"source_path\": \"data/raw/mini_cure_or/test/15631.jpg\", \"source_selection_id\": \"rtv03_shoes_10\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15631.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15631\", \"source_path\": \"data/raw/mini_cure_or/test/15631.jpg\", \"source_selection_id\": \"rtv03_shoes_10\"}\n{\"answer_display\": \"shoes\", \"answer_letter\": \"H\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15631.jpg\", \"label\": \"shoes\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15631\", \"source_path\": \"data/raw/mini_cure_or/test/15631.jpg\", \"source_selection_id\": \"rtv03_shoes_10\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11716.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11716\", \"source_path\": \"data/raw/mini_cure_or/test/11716.jpg\", \"source_selection_id\": \"rtv03_toy_01\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11716.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11716\", \"source_path\": \"data/raw/mini_cure_or/test/11716.jpg\", \"source_selection_id\": \"rtv03_toy_01\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11716.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11716\", \"source_path\": \"data/raw/mini_cure_or/test/11716.jpg\", \"source_selection_id\": \"rtv03_toy_01\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11716.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11716\", \"source_path\": \"data/raw/mini_cure_or/test/11716.jpg\", \"source_selection_id\": \"rtv03_toy_01\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11716.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11716\", \"source_path\": \"data/raw/mini_cure_or/test/11716.jpg\", \"source_selection_id\": \"rtv03_toy_01\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11716.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11716\", \"source_path\": \"data/raw/mini_cure_or/test/11716.jpg\", \"source_selection_id\": \"rtv03_toy_01\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11716.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11716\", \"source_path\": \"data/raw/mini_cure_or/test/11716.jpg\", \"source_selection_id\": \"rtv03_toy_01\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11716.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11716\", \"source_path\": \"data/raw/mini_cure_or/test/11716.jpg\", \"source_selection_id\": \"rtv03_toy_01\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11735.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11735\", \"source_path\": \"data/raw/mini_cure_or/test/11735.jpg\", \"source_selection_id\": \"rtv03_toy_02\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11735.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11735\", \"source_path\": \"data/raw/mini_cure_or/test/11735.jpg\", \"source_selection_id\": \"rtv03_toy_02\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11735.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11735\", \"source_path\": \"data/raw/mini_cure_or/test/11735.jpg\", \"source_selection_id\": \"rtv03_toy_02\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11735.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11735\", \"source_path\": \"data/raw/mini_cure_or/test/11735.jpg\", \"source_selection_id\": \"rtv03_toy_02\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11735.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11735\", \"source_path\": \"data/raw/mini_cure_or/test/11735.jpg\", \"source_selection_id\": \"rtv03_toy_02\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11735.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11735\", \"source_path\": \"data/raw/mini_cure_or/test/11735.jpg\", \"source_selection_id\": \"rtv03_toy_02\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11735.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11735\", \"source_path\": \"data/raw/mini_cure_or/test/11735.jpg\", \"source_selection_id\": \"rtv03_toy_02\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11735.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11735\", \"source_path\": \"data/raw/mini_cure_or/test/11735.jpg\", \"source_selection_id\": \"rtv03_toy_02\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/12027.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__12027\", \"source_path\": \"data/raw/mini_cure_or/test/12027.jpg\", \"source_selection_id\": \"rtv03_toy_03\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/12027.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__12027\", \"source_path\": \"data/raw/mini_cure_or/test/12027.jpg\", \"source_selection_id\": \"rtv03_toy_03\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/12027.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__12027\", \"source_path\": \"data/raw/mini_cure_or/test/12027.jpg\", \"source_selection_id\": \"rtv03_toy_03\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/12027.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__12027\", \"source_path\": \"data/raw/mini_cure_or/test/12027.jpg\", \"source_selection_id\": \"rtv03_toy_03\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/12027.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__12027\", \"source_path\": \"data/raw/mini_cure_or/test/12027.jpg\", \"source_selection_id\": \"rtv03_toy_03\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/12027.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__12027\", \"source_path\": \"data/raw/mini_cure_or/test/12027.jpg\", \"source_selection_id\": \"rtv03_toy_03\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/12027.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__12027\", \"source_path\": \"data/raw/mini_cure_or/test/12027.jpg\", \"source_selection_id\": \"rtv03_toy_03\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/12027.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__12027\", \"source_path\": \"data/raw/mini_cure_or/test/12027.jpg\", \"source_selection_id\": \"rtv03_toy_03\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13864.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13864\", \"source_path\": \"data/raw/mini_cure_or/test/13864.jpg\", \"source_selection_id\": \"rtv03_toy_04\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13864.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13864\", \"source_path\": \"data/raw/mini_cure_or/test/13864.jpg\", \"source_selection_id\": \"rtv03_toy_04\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13864.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13864\", \"source_path\": \"data/raw/mini_cure_or/test/13864.jpg\", \"source_selection_id\": \"rtv03_toy_04\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13864.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13864\", \"source_path\": \"data/raw/mini_cure_or/test/13864.jpg\", \"source_selection_id\": \"rtv03_toy_04\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13864.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13864\", \"source_path\": \"data/raw/mini_cure_or/test/13864.jpg\", \"source_selection_id\": \"rtv03_toy_04\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13864.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13864\", \"source_path\": \"data/raw/mini_cure_or/test/13864.jpg\", \"source_selection_id\": \"rtv03_toy_04\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13864.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13864\", \"source_path\": \"data/raw/mini_cure_or/test/13864.jpg\", \"source_selection_id\": \"rtv03_toy_04\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13864.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13864\", \"source_path\": \"data/raw/mini_cure_or/test/13864.jpg\", \"source_selection_id\": \"rtv03_toy_04\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13931.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13931\", \"source_path\": \"data/raw/mini_cure_or/test/13931.jpg\", \"source_selection_id\": \"rtv03_toy_05\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13931.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13931\", \"source_path\": \"data/raw/mini_cure_or/test/13931.jpg\", \"source_selection_id\": \"rtv03_toy_05\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13931.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13931\", \"source_path\": \"data/raw/mini_cure_or/test/13931.jpg\", \"source_selection_id\": \"rtv03_toy_05\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13931.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13931\", \"source_path\": \"data/raw/mini_cure_or/test/13931.jpg\", \"source_selection_id\": \"rtv03_toy_05\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13931.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13931\", \"source_path\": \"data/raw/mini_cure_or/test/13931.jpg\", \"source_selection_id\": \"rtv03_toy_05\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13931.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13931\", \"source_path\": \"data/raw/mini_cure_or/test/13931.jpg\", \"source_selection_id\": \"rtv03_toy_05\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13931.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13931\", \"source_path\": \"data/raw/mini_cure_or/test/13931.jpg\", \"source_selection_id\": \"rtv03_toy_05\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13931.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13931\", \"source_path\": \"data/raw/mini_cure_or/test/13931.jpg\", \"source_selection_id\": \"rtv03_toy_05\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15247.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15247\", \"source_path\": \"data/raw/mini_cure_or/test/15247.jpg\", \"source_selection_id\": \"rtv03_toy_06\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15247.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15247\", \"source_path\": \"data/raw/mini_cure_or/test/15247.jpg\", \"source_selection_id\": \"rtv03_toy_06\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15247.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15247\", \"source_path\": \"data/raw/mini_cure_or/test/15247.jpg\", \"source_selection_id\": \"rtv03_toy_06\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15247.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15247\", \"source_path\": \"data/raw/mini_cure_or/test/15247.jpg\", \"source_selection_id\": \"rtv03_toy_06\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15247.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15247\", \"source_path\": \"data/raw/mini_cure_or/test/15247.jpg\", \"source_selection_id\": \"rtv03_toy_06\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15247.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15247\", \"source_path\": \"data/raw/mini_cure_or/test/15247.jpg\", \"source_selection_id\": \"rtv03_toy_06\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15247.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15247\", \"source_path\": \"data/raw/mini_cure_or/test/15247.jpg\", \"source_selection_id\": \"rtv03_toy_06\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15247.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15247\", \"source_path\": \"data/raw/mini_cure_or/test/15247.jpg\", \"source_selection_id\": \"rtv03_toy_06\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15321.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15321\", \"source_path\": \"data/raw/mini_cure_or/test/15321.jpg\", \"source_selection_id\": \"rtv03_toy_07\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15321.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15321\", \"source_path\": \"data/raw/mini_cure_or/test/15321.jpg\", \"source_selection_id\": \"rtv03_toy_07\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15321.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15321\", \"source_path\": \"data/raw/mini_cure_or/test/15321.jpg\", \"source_selection_id\": \"rtv03_toy_07\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15321.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15321\", \"source_path\": \"data/raw/mini_cure_or/test/15321.jpg\", \"source_selection_id\": \"rtv03_toy_07\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15321.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15321\", \"source_path\": \"data/raw/mini_cure_or/test/15321.jpg\", \"source_selection_id\": \"rtv03_toy_07\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15321.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15321\", \"source_path\": \"data/raw/mini_cure_or/test/15321.jpg\", \"source_selection_id\": \"rtv03_toy_07\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15321.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15321\", \"source_path\": \"data/raw/mini_cure_or/test/15321.jpg\", \"source_selection_id\": \"rtv03_toy_07\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15321.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15321\", \"source_path\": \"data/raw/mini_cure_or/test/15321.jpg\", \"source_selection_id\": \"rtv03_toy_07\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15600.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15600\", \"source_path\": \"data/raw/mini_cure_or/test/15600.jpg\", \"source_selection_id\": \"rtv03_toy_08\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15600.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15600\", \"source_path\": \"data/raw/mini_cure_or/test/15600.jpg\", \"source_selection_id\": \"rtv03_toy_08\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15600.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15600\", \"source_path\": \"data/raw/mini_cure_or/test/15600.jpg\", \"source_selection_id\": \"rtv03_toy_08\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15600.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15600\", \"source_path\": \"data/raw/mini_cure_or/test/15600.jpg\", \"source_selection_id\": \"rtv03_toy_08\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15600.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15600\", \"source_path\": \"data/raw/mini_cure_or/test/15600.jpg\", \"source_selection_id\": \"rtv03_toy_08\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15600.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15600\", \"source_path\": \"data/raw/mini_cure_or/test/15600.jpg\", \"source_selection_id\": \"rtv03_toy_08\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15600.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15600\", \"source_path\": \"data/raw/mini_cure_or/test/15600.jpg\", \"source_selection_id\": \"rtv03_toy_08\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15600.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15600\", \"source_path\": \"data/raw/mini_cure_or/test/15600.jpg\", \"source_selection_id\": \"rtv03_toy_08\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/16118.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__16118\", \"source_path\": \"data/raw/mini_cure_or/test/16118.jpg\", \"source_selection_id\": \"rtv03_toy_09\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/16118.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__16118\", \"source_path\": \"data/raw/mini_cure_or/test/16118.jpg\", \"source_selection_id\": \"rtv03_toy_09\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/16118.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__16118\", \"source_path\": \"data/raw/mini_cure_or/test/16118.jpg\", \"source_selection_id\": \"rtv03_toy_09\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/16118.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__16118\", \"source_path\": \"data/raw/mini_cure_or/test/16118.jpg\", \"source_selection_id\": \"rtv03_toy_09\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/16118.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__16118\", \"source_path\": \"data/raw/mini_cure_or/test/16118.jpg\", \"source_selection_id\": \"rtv03_toy_09\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/16118.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__16118\", \"source_path\": \"data/raw/mini_cure_or/test/16118.jpg\", \"source_selection_id\": \"rtv03_toy_09\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/16118.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__16118\", \"source_path\": \"data/raw/mini_cure_or/test/16118.jpg\", \"source_selection_id\": \"rtv03_toy_09\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/16118.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__16118\", \"source_path\": \"data/raw/mini_cure_or/test/16118.jpg\", \"source_selection_id\": \"rtv03_toy_09\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/16499.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__16499\", \"source_path\": \"data/raw/mini_cure_or/test/16499.jpg\", \"source_selection_id\": \"rtv03_toy_10\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/16499.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__16499\", \"source_path\": \"data/raw/mini_cure_or/test/16499.jpg\", \"source_selection_id\": \"rtv03_toy_10\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/16499.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__16499\", \"source_path\": \"data/raw/mini_cure_or/test/16499.jpg\", \"source_selection_id\": \"rtv03_toy_10\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/16499.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__16499\", \"source_path\": \"data/raw/mini_cure_or/test/16499.jpg\", \"source_selection_id\": \"rtv03_toy_10\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/16499.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__16499\", \"source_path\": \"data/raw/mini_cure_or/test/16499.jpg\", \"source_selection_id\": \"rtv03_toy_10\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/16499.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__16499\", \"source_path\": \"data/raw/mini_cure_or/test/16499.jpg\", \"source_selection_id\": \"rtv03_toy_10\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/16499.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__16499\", \"source_path\": \"data/raw/mini_cure_or/test/16499.jpg\", \"source_selection_id\": \"rtv03_toy_10\"}\n{\"answer_display\": \"toy\", \"answer_letter\": \"I\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/16499.jpg\", \"label\": \"toy\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__16499\", \"source_path\": \"data/raw/mini_cure_or/test/16499.jpg\", \"source_selection_id\": \"rtv03_toy_10\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/10457.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__10457\", \"source_path\": \"data/raw/mini_cure_or/test/10457.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_01\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/10457.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__10457\", \"source_path\": \"data/raw/mini_cure_or/test/10457.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_01\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/10457.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__10457\", \"source_path\": \"data/raw/mini_cure_or/test/10457.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_01\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/10457.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__10457\", \"source_path\": \"data/raw/mini_cure_or/test/10457.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_01\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/10457.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__10457\", \"source_path\": \"data/raw/mini_cure_or/test/10457.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_01\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/10457.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__10457\", \"source_path\": \"data/raw/mini_cure_or/test/10457.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_01\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/10457.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__10457\", \"source_path\": \"data/raw/mini_cure_or/test/10457.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_01\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/10457.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__10457\", \"source_path\": \"data/raw/mini_cure_or/test/10457.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_01\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11437.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11437\", \"source_path\": \"data/raw/mini_cure_or/test/11437.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_02\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11437.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11437\", \"source_path\": \"data/raw/mini_cure_or/test/11437.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_02\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11437.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11437\", \"source_path\": \"data/raw/mini_cure_or/test/11437.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_02\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11437.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11437\", \"source_path\": \"data/raw/mini_cure_or/test/11437.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_02\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11437.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11437\", \"source_path\": \"data/raw/mini_cure_or/test/11437.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_02\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11437.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11437\", \"source_path\": \"data/raw/mini_cure_or/test/11437.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_02\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11437.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11437\", \"source_path\": \"data/raw/mini_cure_or/test/11437.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_02\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11437.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11437\", \"source_path\": \"data/raw/mini_cure_or/test/11437.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_02\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/11747.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__11747\", \"source_path\": \"data/raw/mini_cure_or/test/11747.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_03\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/11747.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__11747\", \"source_path\": \"data/raw/mini_cure_or/test/11747.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_03\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/11747.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__11747\", \"source_path\": \"data/raw/mini_cure_or/test/11747.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_03\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/11747.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__11747\", \"source_path\": \"data/raw/mini_cure_or/test/11747.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_03\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/11747.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__11747\", \"source_path\": \"data/raw/mini_cure_or/test/11747.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_03\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/11747.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__11747\", \"source_path\": \"data/raw/mini_cure_or/test/11747.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_03\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/11747.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__11747\", \"source_path\": \"data/raw/mini_cure_or/test/11747.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_03\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/11747.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__11747\", \"source_path\": \"data/raw/mini_cure_or/test/11747.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_03\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/13638.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__13638\", \"source_path\": \"data/raw/mini_cure_or/test/13638.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_04\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/13638.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__13638\", \"source_path\": \"data/raw/mini_cure_or/test/13638.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_04\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/13638.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__13638\", \"source_path\": \"data/raw/mini_cure_or/test/13638.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_04\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/13638.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__13638\", \"source_path\": \"data/raw/mini_cure_or/test/13638.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_04\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/13638.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__13638\", \"source_path\": \"data/raw/mini_cure_or/test/13638.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_04\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/13638.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__13638\", \"source_path\": \"data/raw/mini_cure_or/test/13638.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_04\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/13638.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__13638\", \"source_path\": \"data/raw/mini_cure_or/test/13638.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_04\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/13638.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__13638\", \"source_path\": \"data/raw/mini_cure_or/test/13638.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_04\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/14003.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__14003\", \"source_path\": \"data/raw/mini_cure_or/test/14003.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_05\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/14003.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__14003\", \"source_path\": \"data/raw/mini_cure_or/test/14003.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_05\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/14003.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__14003\", \"source_path\": \"data/raw/mini_cure_or/test/14003.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_05\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/14003.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__14003\", \"source_path\": \"data/raw/mini_cure_or/test/14003.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_05\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/14003.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__14003\", \"source_path\": \"data/raw/mini_cure_or/test/14003.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_05\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/14003.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__14003\", \"source_path\": \"data/raw/mini_cure_or/test/14003.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_05\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/14003.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__14003\", \"source_path\": \"data/raw/mini_cure_or/test/14003.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_05\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/14003.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__14003\", \"source_path\": \"data/raw/mini_cure_or/test/14003.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_05\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/14029.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__14029\", \"source_path\": \"data/raw/mini_cure_or/test/14029.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_06\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/14029.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__14029\", \"source_path\": \"data/raw/mini_cure_or/test/14029.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_06\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/14029.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__14029\", \"source_path\": \"data/raw/mini_cure_or/test/14029.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_06\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/14029.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__14029\", \"source_path\": \"data/raw/mini_cure_or/test/14029.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_06\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/14029.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__14029\", \"source_path\": \"data/raw/mini_cure_or/test/14029.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_06\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/14029.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__14029\", \"source_path\": \"data/raw/mini_cure_or/test/14029.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_06\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/14029.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__14029\", \"source_path\": \"data/raw/mini_cure_or/test/14029.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_06\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/14029.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__14029\", \"source_path\": \"data/raw/mini_cure_or/test/14029.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_06\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/14751.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__14751\", \"source_path\": \"data/raw/mini_cure_or/test/14751.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_07\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/14751.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__14751\", \"source_path\": \"data/raw/mini_cure_or/test/14751.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_07\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/14751.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__14751\", \"source_path\": \"data/raw/mini_cure_or/test/14751.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_07\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/14751.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__14751\", \"source_path\": \"data/raw/mini_cure_or/test/14751.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_07\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/14751.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__14751\", \"source_path\": \"data/raw/mini_cure_or/test/14751.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_07\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/14751.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__14751\", \"source_path\": \"data/raw/mini_cure_or/test/14751.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_07\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/14751.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__14751\", \"source_path\": \"data/raw/mini_cure_or/test/14751.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_07\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/14751.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__14751\", \"source_path\": \"data/raw/mini_cure_or/test/14751.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_07\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/14901.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__14901\", \"source_path\": \"data/raw/mini_cure_or/test/14901.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_08\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/14901.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__14901\", \"source_path\": \"data/raw/mini_cure_or/test/14901.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_08\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/14901.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__14901\", \"source_path\": \"data/raw/mini_cure_or/test/14901.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_08\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/14901.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__14901\", \"source_path\": \"data/raw/mini_cure_or/test/14901.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_08\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/14901.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__14901\", \"source_path\": \"data/raw/mini_cure_or/test/14901.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_08\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/14901.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__14901\", \"source_path\": \"data/raw/mini_cure_or/test/14901.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_08\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/14901.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__14901\", \"source_path\": \"data/raw/mini_cure_or/test/14901.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_08\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/14901.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__14901\", \"source_path\": \"data/raw/mini_cure_or/test/14901.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_08\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15604.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15604\", \"source_path\": \"data/raw/mini_cure_or/test/15604.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_09\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15604.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15604\", \"source_path\": \"data/raw/mini_cure_or/test/15604.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_09\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15604.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15604\", \"source_path\": \"data/raw/mini_cure_or/test/15604.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_09\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15604.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15604\", \"source_path\": \"data/raw/mini_cure_or/test/15604.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_09\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15604.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15604\", \"source_path\": \"data/raw/mini_cure_or/test/15604.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_09\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15604.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15604\", \"source_path\": \"data/raw/mini_cure_or/test/15604.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_09\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15604.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15604\", \"source_path\": \"data/raw/mini_cure_or/test/15604.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_09\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15604.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15604\", \"source_path\": \"data/raw/mini_cure_or/test/15604.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_09\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_01/15877.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_01__15877\", \"source_path\": \"data/raw/mini_cure_or/test/15877.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_10\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/messenger_upload_download/rep_02/15877.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"messenger_upload_download\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__messenger_upload_download__rep_02__15877\", \"source_path\": \"data/raw/mini_cure_or/test/15877.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_10\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_01/15877.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_01__15877\", \"source_path\": \"data/raw/mini_cure_or/test/15877.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_10\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/phone_screenshot_resave/rep_02/15877.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"phone_screenshot_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__phone_screenshot_resave__rep_02__15877\", \"source_path\": \"data/raw/mini_cure_or/test/15877.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_10\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_01/15877.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_01__15877\", \"source_path\": \"data/raw/mini_cure_or/test/15877.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_10\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/video_call_frame_capture/rep_02/15877.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"video_call_frame_capture\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__video_call_frame_capture__rep_02__15877\", \"source_path\": \"data/raw/mini_cure_or/test/15877.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_10\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_01/15877.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_01\", \"sample_id\": \"real_transfer__social_app_resave__rep_01__15877\", \"source_path\": \"data/raw/mini_cure_or/test/15877.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_10\"}\n{\"answer_display\": \"training marker cone\", \"answer_letter\": \"J\", \"expected_response_format\": \"single_letter\", \"family\": \"real_transfer\", \"image_path\": \"data/real_transfer/v03/images/social_app_resave/rep_02/15877.jpg\", \"label\": \"training_marker_cone\", \"options\": [{\"display\": \"baseball\", \"label\": \"baseball\", \"letter\": \"A\"}, {\"display\": \"calcium bottle\", \"label\": \"calcium_bottle\", \"letter\": \"B\"}, {\"display\": \"Canon camera\", \"label\": \"canon_camera\", \"letter\": \"C\"}, {\"display\": \"DYMO label maker\", \"label\": \"dymo_label_maker\", \"letter\": \"D\"}, {\"display\": \"hair brush\", \"label\": \"hair_brush\", \"letter\": \"E\"}, {\"display\": \"LG cell phone\", \"label\": \"lg_cell_phone\", \"letter\": \"F\"}, {\"display\": \"pan\", \"label\": \"pan\", \"letter\": \"G\"}, {\"display\": \"shoes\", \"label\": \"shoes\", \"letter\": \"H\"}, {\"display\": \"toy\", \"label\": \"toy\", \"letter\": \"I\"}, {\"display\": \"training marker cone\", \"label\": \"training_marker_cone\", \"letter\": \"J\"}], \"prompt\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\\n\\nOptions:\\nA. baseball\\nB. calcium bottle\\nC. Canon camera\\nD. DYMO label maker\\nE. hair brush\\nF. LG cell phone\\nG. pan\\nH. shoes\\nI. toy\\nJ. training marker cone\", \"question\": \"Which object category is shown in the image? Answer with exactly one letter from the options.\", \"recipe\": \"social_app_resave\", \"repeat_id\": \"rep_02\", \"sample_id\": \"real_transfer__social_app_resave__rep_02__15877\", \"source_path\": \"data/raw/mini_cure_or/test/15877.jpg\", \"source_selection_id\": \"rtv03_training_marker_cone_10\"}\n", "scripts/run_hf_vlm.py": "#!/usr/bin/env python3\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport hashlib\nimport json\nimport os\nimport time\nfrom datetime import UTC, datetime\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\nDEFAULT_SYSTEM_PROMPT = (\n    \"You are evaluating object recognition robustness. \"\n    \"Answer with exactly one option letter and no explanation.\"\n)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=\"Run a cached local Hugging Face VLM over a CURE-OR++ prompt pack.\")\n    parser.add_argument(\"--prompt-pack\", default=\"reports/vlm_api_track_v01_prompt_pack.jsonl\")\n    parser.add_argument(\"--output\", required=True, help=\"Sanitized response JSONL output path.\")\n    parser.add_argument(\"--cache-dir\", default=\"data/vlm_api_cache/huggingface\")\n    parser.add_argument(\"--provider\", default=\"huggingface\")\n    parser.add_argument(\"--model\", default=\"HuggingFaceTB/SmolVLM2-500M-Video-Instruct\")\n    parser.add_argument(\"--model-version\", default=\"\", help=\"Revision, commit, or exact local model version if known.\")\n    parser.add_argument(\"--revision\", default=None)\n    parser.add_argument(\"--processor-kwargs-json\", default=\"{}\", help=\"JSON object passed to AutoProcessor.from_pretrained.\")\n    parser.add_argument(\"--model-kwargs-json\", default=\"{}\", help=\"JSON object merged into model from_pretrained kwargs.\")\n    parser.add_argument(\"--generate-kwargs-json\", default=\"{}\", help=\"JSON object merged into model.generate kwargs.\")\n    parser.add_argument(\n        \"--image-max-side\",\n        type=int,\n        default=0,\n        help=\"If > 0, downscale each input image so its largest side is at most this many pixels before preprocessing.\",\n    )\n    parser.add_argument(\"--device-map\", default=\"\", help=\"Optional Transformers device_map value, for example 'auto'.\")\n    parser.add_argument(\n        \"--model-class\",\n        choices=[\n            \"AutoModelForImageTextToText\",\n            \"AutoModelForVision2Seq\",\n            \"LlavaOnevisionForConditionalGeneration\",\n            \"Qwen2_5_VLForConditionalGeneration\",\n        ],\n        default=\"AutoModelForImageTextToText\",\n    )\n    parser.add_argument(\n        \"--generation-backend\",\n        choices=[\"chat_template\", \"qwen_vl_utils\"],\n        default=\"chat_template\",\n        help=\"How to turn chat messages into model inputs.\",\n    )\n    parser.add_argument(\"--trust-remote-code\", action=\"store_true\")\n    parser.add_argument(\"--device\", choices=[\"auto\", \"cpu\", \"cuda\", \"mps\"], default=\"auto\")\n    parser.add_argument(\"--dtype\", choices=[\"auto\", \"float32\", \"float16\", \"bfloat16\"], default=\"auto\")\n    parser.add_argument(\n        \"--image-content-key\",\n        choices=[\"path\", \"url\", \"url_path\", \"image\", \"image_path\"],\n        default=\"path\",\n        help=\"How to reference local images inside the Transformers chat template.\",\n    )\n    parser.add_argument(\"--max-tokens\", type=int, default=8)\n    parser.add_argument(\"--request-sleep\", type=float, default=0.0)\n    parser.add_argument(\"--limit\", type=int)\n    parser.add_argument(\"--family\", choices=[\"clean\", \"real_transfer\"])\n    parser.add_argument(\"--recipe\", action=\"append\", help=\"Restrict to one or more recipes.\")\n    parser.add_argument(\"--sample-id\", action=\"append\", help=\"Restrict to one or more sample IDs.\")\n    parser.add_argument(\"--system-prompt\", default=DEFAULT_SYSTEM_PROMPT)\n    parser.add_argument(\"--force\", action=\"store_true\", help=\"Overwrite output and ignore existing output rows.\")\n    parser.add_argument(\"--dry-run\", action=\"store_true\", help=\"Validate inputs and print planned work without loading a model.\")\n    parser.add_argument(\n        \"--mock-oracle\",\n        action=\"store_true\",\n        help=\"Write ground-truth letters as responses. For runner/evaluator smoke tests only.\",\n    )\n    args = parser.parse_args()\n    args.processor_kwargs = parse_json_object(args.processor_kwargs_json, \"--processor-kwargs-json\")\n    args.model_kwargs = parse_json_object(args.model_kwargs_json, \"--model-kwargs-json\")\n    args.generate_kwargs = parse_json_object(args.generate_kwargs_json, \"--generate-kwargs-json\")\n\n    prompt_rows = select_rows(load_jsonl(resolve_project_path(args.prompt_pack)), args)\n    if args.limit is not None:\n        prompt_rows = prompt_rows[: args.limit]\n\n    missing_images = [row[\"image_path\"] for row in prompt_rows if not resolve_project_path(row[\"image_path\"]).exists()]\n    if missing_images:\n        raise FileNotFoundError(f\"Missing image files: {missing_images[:5]} total={len(missing_images)}\")\n\n    output_path = resolve_project_path(args.output)\n    cache_dir = resolve_project_path(args.cache_dir)\n    completed_ids = set() if args.force else load_completed_sample_ids(output_path)\n    pending_rows = [row for row in prompt_rows if row[\"sample_id\"] not in completed_ids]\n\n    print(f\"Prompt rows selected: {len(prompt_rows)}\")\n    print(f\"Already completed: {len(completed_ids & {row['sample_id'] for row in prompt_rows})}\")\n    print(f\"Pending rows: {len(pending_rows)}\")\n    print(f\"Backend: transformers\")\n    print(f\"Provider/model: {args.provider}/{args.model}\")\n    print(f\"Model class: {args.model_class}\")\n    print(f\"Output: {output_path}\")\n    print(f\"Cache dir: {cache_dir}\")\n\n    if args.dry_run:\n        return 0\n\n    if args.force and output_path.exists():\n        output_path.unlink()\n\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    cache_dir.mkdir(parents=True, exist_ok=True)\n\n    runtime = None if args.mock_oracle else load_runtime(args)\n\n    written = 0\n    with output_path.open(\"a\", encoding=\"utf-8\") as output_handle:\n        for index, row in enumerate(pending_rows, start=1):\n            result = run_row(row, args, cache_dir, runtime)\n            output_handle.write(json.dumps(result, sort_keys=True) + \"\\n\")\n            output_handle.flush()\n            written += 1\n            clear_runtime_memory(runtime)\n            if index % 10 == 0 or index == len(pending_rows):\n                print(f\"Processed {index}/{len(pending_rows)} pending rows\")\n            if args.request_sleep > 0 and index < len(pending_rows):\n                time.sleep(args.request_sleep)\n\n    print(f\"Wrote rows: {written}\")\n    return 0\n\n\ndef load_runtime(args: argparse.Namespace) -> dict:\n    try:\n        import torch\n        import transformers\n    except ImportError as exc:\n        raise RuntimeError(\"Missing local VLM dependencies. Install torch, transformers, and pillow.\") from exc\n\n    model_class = getattr(transformers, args.model_class)\n    processor = transformers.AutoProcessor.from_pretrained(\n        args.model,\n        revision=args.revision,\n        trust_remote_code=args.trust_remote_code,\n        **args.processor_kwargs,\n    )\n    device = choose_device(args.device, torch)\n    dtype = choose_dtype(args.dtype, device, torch)\n\n    load_kwargs = {\n        \"revision\": args.revision,\n        \"trust_remote_code\": args.trust_remote_code,\n    }\n    load_kwargs.update(args.model_kwargs)\n    if dtype is not None:\n        load_kwargs[\"torch_dtype\"] = dtype\n    if args.device_map:\n        load_kwargs[\"device_map\"] = args.device_map\n    try:\n        model = model_class.from_pretrained(args.model, **load_kwargs)\n    except TypeError:\n        if \"torch_dtype\" in load_kwargs:\n            load_kwargs[\"dtype\"] = load_kwargs.pop(\"torch_dtype\")\n        model = model_class.from_pretrained(args.model, **load_kwargs)\n\n    if not args.device_map:\n        model = model.to(device)\n    model.eval()\n    print(f\"Loaded model on device={device} dtype={dtype} device_map={args.device_map or 'none'}\")\n    return {\n        \"torch\": torch,\n        \"processor\": processor,\n        \"model\": model,\n        \"device\": device,\n        \"dtype\": dtype,\n    }\n\n\ndef run_row(row: dict, args: argparse.Namespace, cache_dir: Path, runtime: dict | None) -> dict:\n    image_path = resolve_project_path(row[\"image_path\"])\n    image_hash = sha256_file(image_path)\n    prompt_hash = sha256_text(row[\"prompt\"])\n    request_fingerprint = build_request_fingerprint(\n        row=row,\n        args=args,\n        image_hash=image_hash,\n        prompt_hash=prompt_hash,\n    )\n    cache_key = sha256_text(json.dumps(request_fingerprint, sort_keys=True))\n    cache_path = cache_dir / safe_slug(args.provider) / safe_slug(args.model) / f\"{cache_key}.json\"\n\n    if cache_path.exists():\n        cached = json.loads(cache_path.read_text(encoding=\"utf-8\"))\n        return build_output_row(row, args, cached, cache_key, image_hash, prompt_hash, from_cache=True)\n\n    if args.mock_oracle:\n        response_text = row[\"answer_letter\"]\n    else:\n        if runtime is None:\n            raise RuntimeError(\"Runtime is required unless --mock-oracle is set.\")\n        response_text = generate_response(row, args, image_path, runtime)\n\n    cached = {\n        \"cached_at_utc\": datetime.now(UTC).isoformat(),\n        \"provider\": args.provider,\n        \"model_id\": args.model,\n        \"model_version\": args.model_version,\n        \"endpoint\": \"local_transformers\",\n        \"response_text\": response_text,\n        \"request_fingerprint\": request_fingerprint,\n    }\n    cache_path.parent.mkdir(parents=True, exist_ok=True)\n    cache_path.write_text(json.dumps(cached, indent=2, sort_keys=True) + \"\\n\", encoding=\"utf-8\")\n    return build_output_row(row, args, cached, cache_key, image_hash, prompt_hash, from_cache=False)\n\n\ndef generate_response(row: dict, args: argparse.Namespace, image_path: Path, runtime: dict) -> str:\n    torch = runtime[\"torch\"]\n    processor = runtime[\"processor\"]\n    model = runtime[\"model\"]\n    device = runtime[\"device\"]\n    dtype = runtime[\"dtype\"]\n\n    image_source = prepare_image_source(image_path, args)\n    messages = build_messages(row, args, image_source)\n    if args.generation_backend == \"qwen_vl_utils\":\n        try:\n            from qwen_vl_utils import process_vision_info\n        except ImportError as exc:\n            raise RuntimeError(\"Missing qwen-vl-utils. Install qwen-vl-utils for Qwen2.5-VL runs.\") from exc\n        prompt_text = processor.apply_chat_template(\n            messages,\n            add_generation_prompt=True,\n            tokenize=False,\n        )\n        image_inputs, video_inputs = process_vision_info(messages)\n        inputs = processor(\n            text=[prompt_text],\n            images=image_inputs,\n            videos=video_inputs,\n            padding=True,\n            return_tensors=\"pt\",\n        )\n    else:\n        inputs = processor.apply_chat_template(\n            messages,\n            add_generation_prompt=True,\n            tokenize=True,\n            return_dict=True,\n            return_tensors=\"pt\",\n        )\n    inputs = move_inputs(inputs, device=device, dtype=dtype, torch=torch)\n    input_length = inputs[\"input_ids\"].shape[-1]\n    with torch.inference_mode():\n        generate_kwargs = {\n            \"do_sample\": False,\n            \"max_new_tokens\": args.max_tokens,\n        }\n        generate_kwargs.update(args.generate_kwargs)\n        generated_ids = model.generate(\n            **inputs,\n            **generate_kwargs,\n        )\n    new_tokens = generated_ids[:, input_length:]\n    decoded = processor.batch_decode(new_tokens, skip_special_tokens=True)\n    return decoded[0].strip() if decoded else \"\"\n\n\ndef prepare_image_source(image_path: Path, args: argparse.Namespace):\n    if args.image_max_side <= 0:\n        return image_path\n\n    try:\n        from PIL import Image, ImageOps\n    except ImportError as exc:\n        raise RuntimeError(\"Missing Pillow. Install pillow to use --image-max-side.\") from exc\n\n    with Image.open(image_path) as source_image:\n        image = ImageOps.exif_transpose(source_image).convert(\"RGB\")\n        width, height = image.size\n        largest_side = max(width, height)\n        if largest_side <= args.image_max_side:\n            return image.copy()\n        scale = args.image_max_side / float(largest_side)\n        new_size = (\n            max(1, int(round(width * scale))),\n            max(1, int(round(height * scale))),\n        )\n        resampling = getattr(Image, \"Resampling\", Image).LANCZOS\n        return image.resize(new_size, resampling)\n\n\ndef build_messages(row: dict, args: argparse.Namespace, image_source) -> list[dict]:\n    if isinstance(image_source, Path):\n        image_value = str(image_source.resolve())\n        image_uri = image_source.resolve().as_uri()\n        image_part: dict[str, object]\n        if args.image_content_key == \"url\":\n            image_part = {\"type\": \"image\", \"url\": image_uri}\n        elif args.image_content_key == \"url_path\":\n            image_part = {\"type\": \"image\", \"url\": image_value}\n        elif args.image_content_key == \"image\":\n            image_part = {\"type\": \"image\", \"image\": image_uri}\n        elif args.image_content_key == \"image_path\":\n            image_part = {\"type\": \"image\", \"image\": image_value}\n        else:\n            image_part = {\"type\": \"image\", \"path\": image_value}\n    else:\n        image_part = {\"type\": \"image\", \"image\": image_source}\n    content = [\n        image_part,\n        {\"type\": \"text\", \"text\": row[\"prompt\"]},\n    ]\n    messages = [{\"role\": \"user\", \"content\": content}]\n    if args.system_prompt:\n        messages.insert(0, {\"role\": \"system\", \"content\": [{\"type\": \"text\", \"text\": args.system_prompt}]})\n    return messages\n\n\ndef move_inputs(inputs: dict, *, device: str, dtype, torch):\n    moved = {}\n    for key, value in inputs.items():\n        if hasattr(value, \"to\"):\n            if torch.is_floating_point(value) and dtype is not None:\n                moved[key] = value.to(device=device, dtype=dtype)\n            else:\n                moved[key] = value.to(device=device)\n        else:\n            moved[key] = value\n    return moved\n\n\ndef build_output_row(\n    row: dict,\n    args: argparse.Namespace,\n    cached: dict,\n    cache_key: str,\n    image_hash: str,\n    prompt_hash: str,\n    *,\n    from_cache: bool,\n) -> dict:\n    return {\n        \"sample_id\": row[\"sample_id\"],\n        \"provider\": args.provider,\n        \"model_id\": args.model,\n        \"model_version\": args.model_version,\n        \"endpoint\": \"local_transformers\",\n        \"response_text\": cached[\"response_text\"],\n        \"cache_key\": cache_key,\n        \"from_cache\": from_cache,\n        \"image_sha256\": image_hash,\n        \"prompt_sha256\": prompt_hash,\n        \"request_date_utc\": cached[\"cached_at_utc\"],\n        \"temperature\": 0.0,\n        \"max_tokens\": args.max_tokens,\n    }\n\n\ndef build_request_fingerprint(\n    *,\n    row: dict,\n    args: argparse.Namespace,\n    image_hash: str,\n    prompt_hash: str,\n) -> dict:\n    return {\n        \"sample_id\": row[\"sample_id\"],\n        \"provider\": args.provider,\n        \"model_id\": args.model,\n        \"model_version\": args.model_version,\n        \"revision\": args.revision or \"\",\n        \"endpoint\": \"local_transformers\",\n        \"model_class\": args.model_class,\n        \"generation_backend\": args.generation_backend,\n        \"image_content_key\": args.image_content_key,\n        \"image_max_side\": args.image_max_side,\n        \"image_sha256\": image_hash,\n        \"prompt_sha256\": prompt_hash,\n        \"max_tokens\": args.max_tokens,\n        \"system_prompt_sha256\": sha256_text(args.system_prompt or \"\"),\n    }\n\n\ndef choose_device(requested: str, torch) -> str:\n    if requested != \"auto\":\n        return requested\n    if torch.cuda.is_available():\n        return \"cuda\"\n    if getattr(torch.backends, \"mps\", None) is not None and torch.backends.mps.is_available():\n        return \"mps\"\n    return \"cpu\"\n\n\ndef choose_dtype(requested: str, device: str, torch):\n    if requested == \"float32\":\n        return torch.float32\n    if requested == \"float16\":\n        return torch.float16\n    if requested == \"bfloat16\":\n        return torch.bfloat16\n    if requested == \"auto\":\n        if device == \"cuda\":\n            if hasattr(torch.cuda, \"is_bf16_supported\") and torch.cuda.is_bf16_supported():\n                return torch.bfloat16\n            return torch.float16\n        return torch.float32\n    return None\n\n\ndef select_rows(rows: list[dict], args: argparse.Namespace) -> list[dict]:\n    selected = rows\n    if args.family:\n        selected = [row for row in selected if row[\"family\"] == args.family]\n    if args.recipe:\n        recipes = set(args.recipe)\n        selected = [row for row in selected if row[\"recipe\"] in recipes]\n    if args.sample_id:\n        sample_ids = set(args.sample_id)\n        selected = [row for row in selected if row[\"sample_id\"] in sample_ids]\n    return selected\n\n\ndef load_completed_sample_ids(path: Path) -> set[str]:\n    if not path.exists():\n        return set()\n    return {\n        row[\"sample_id\"]\n        for row in load_jsonl(path)\n        if row.get(\"sample_id\")\n    }\n\n\ndef load_jsonl(path: Path) -> list[dict]:\n    rows = []\n    with path.open(\"r\", encoding=\"utf-8\") as handle:\n        for line_number, line in enumerate(handle, start=1):\n            if line.strip():\n                try:\n                    rows.append(json.loads(line))\n                except json.JSONDecodeError as exc:\n                    raise ValueError(f\"Invalid JSON on {path}:{line_number}\") from exc\n    return rows\n\n\ndef parse_json_object(value: str, argument_name: str) -> dict:\n    try:\n        parsed = json.loads(value)\n    except json.JSONDecodeError as exc:\n        raise ValueError(f\"{argument_name} must be valid JSON\") from exc\n    if not isinstance(parsed, dict):\n        raise ValueError(f\"{argument_name} must be a JSON object\")\n    return parsed\n\n\ndef clear_runtime_memory(runtime: dict | None) -> None:\n    if runtime is None:\n        return\n    torch = runtime.get(\"torch\")\n    if torch is None:\n        return\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open(\"rb\") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef sha256_text(text: str) -> str:\n    return hashlib.sha256(text.encode(\"utf-8\")).hexdigest()\n\n\ndef safe_slug(value: str) -> str:\n    return \"\".join(character if character.isalnum() or character in \"-._\" else \"_\" for character in value)[:120]\n\n\ndef resolve_project_path(path: str | Path) -> Path:\n    candidate = Path(path)\n    if candidate.is_absolute():\n        return candidate\n    return ROOT / candidate\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n", "scripts/evaluate_vlm_response_pack.py": "#!/usr/bin/env python3\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport random\nimport re\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\nBOOTSTRAP_SAMPLES = 5000\nBOOTSTRAP_SEED = 20260616\n\nABSTENTION_PATTERNS = [\n    \"cannot determine\",\n    \"can't determine\",\n    \"can not determine\",\n    \"unable to determine\",\n    \"not enough information\",\n    \"not sure\",\n    \"unclear\",\n    \"unknown\",\n    \"i don't know\",\n]\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=\"Evaluate sanitized VLM/API responses against a CURE-OR++ prompt pack.\")\n    parser.add_argument(\"--prompt-pack\", default=\"reports/vlm_api_track_v01_prompt_pack.jsonl\")\n    parser.add_argument(\"--responses\", required=True, help=\"JSONL with sample_id, model_id, and response_text/output_text/answer_text.\")\n    parser.add_argument(\"--model-summary\", default=\"reports/vlm_api_track_v01_model_summary.csv\")\n    parser.add_argument(\"--recipe-table\", default=\"reports/vlm_api_track_v01_recipe_table.csv\")\n    parser.add_argument(\"--label-table\", default=\"reports/vlm_api_track_v01_label_table.csv\")\n    parser.add_argument(\"--audit-table\", default=\"reports/vlm_api_track_v01_response_audit.csv\")\n    args = parser.parse_args()\n\n    prompt_rows = load_jsonl(resolve_project_path(args.prompt_pack))\n    response_rows = load_jsonl(resolve_project_path(args.responses))\n    prompt_by_id = {row[\"sample_id\"]: row for row in prompt_rows}\n    allowed_letters = sorted({option[\"letter\"] for row in prompt_rows for option in row[\"options\"]})\n\n    joined_rows = join_responses(prompt_by_id, response_rows, allowed_letters)\n    if not joined_rows:\n        raise ValueError(\"No joined response rows. Check sample_id values in the response JSONL.\")\n\n    model_summary = build_model_summary(joined_rows)\n    recipe_table = build_recipe_table(joined_rows)\n    label_table = build_label_table(joined_rows)\n    audit_rows = build_audit_rows(joined_rows)\n\n    write_csv(resolve_project_path(args.model_summary), model_summary, MODEL_SUMMARY_FIELDS)\n    write_csv(resolve_project_path(args.recipe_table), recipe_table, RECIPE_TABLE_FIELDS)\n    write_csv(resolve_project_path(args.label_table), label_table, LABEL_TABLE_FIELDS)\n    write_csv(resolve_project_path(args.audit_table), audit_rows, AUDIT_FIELDS)\n\n    print(f\"Prompt rows: {len(prompt_rows)}\")\n    print(f\"Response rows: {len(response_rows)}\")\n    print(f\"Joined rows: {len(joined_rows)}\")\n    print(f\"Models: {len({row['model_id'] for row in joined_rows})}\")\n    print(f\"Model summary: {resolve_project_path(args.model_summary)}\")\n    print(f\"Recipe table: {resolve_project_path(args.recipe_table)}\")\n    print(f\"Label table: {resolve_project_path(args.label_table)}\")\n    print(f\"Audit table: {resolve_project_path(args.audit_table)}\")\n    return 0\n\n\nMODEL_SUMMARY_FIELDS = [\n    \"model_id\",\n    \"provider\",\n    \"model_version\",\n    \"clean_n\",\n    \"clean_accuracy\",\n    \"clean_unparseable_rate\",\n    \"clean_abstention_rate\",\n    \"real_n\",\n    \"real_accuracy\",\n    \"real_unparseable_rate\",\n    \"real_abstention_rate\",\n    \"accuracy_drop_vs_clean\",\n]\n\nRECIPE_TABLE_FIELDS = [\n    \"model_id\",\n    \"provider\",\n    \"model_version\",\n    \"recipe\",\n    \"n\",\n    \"accuracy\",\n    \"accuracy_ci_low\",\n    \"accuracy_ci_high\",\n    \"unparseable_rate\",\n    \"abstention_rate\",\n    \"accuracy_drop_vs_clean\",\n    \"accuracy_drop_ci_low\",\n    \"accuracy_drop_ci_high\",\n]\n\nLABEL_TABLE_FIELDS = [\n    \"model_id\",\n    \"provider\",\n    \"model_version\",\n    \"label\",\n    \"clean_n\",\n    \"clean_accuracy\",\n    \"real_n\",\n    \"real_accuracy\",\n    \"real_unparseable_rate\",\n    \"accuracy_drop_vs_clean\",\n]\n\nAUDIT_FIELDS = [\n    \"model_id\",\n    \"provider\",\n    \"model_version\",\n    \"sample_id\",\n    \"family\",\n    \"recipe\",\n    \"label\",\n    \"answer_letter\",\n    \"parsed_letter\",\n    \"is_correct\",\n    \"is_unparseable\",\n    \"is_abstention\",\n    \"response_text\",\n]\n\n\ndef join_responses(prompt_by_id: dict[str, dict], response_rows: list[dict], allowed_letters: list[str]) -> list[dict]:\n    duplicate_counts = Counter(row.get(\"sample_id\", \"\") for row in response_rows)\n    duplicates = sorted(sample_id for sample_id, count in duplicate_counts.items() if sample_id and count > 1)\n    if duplicates:\n        raise ValueError(f\"Duplicate response sample_id values: {duplicates[:5]}\")\n\n    output = []\n    missing = []\n    for response in response_rows:\n        sample_id = response.get(\"sample_id\")\n        prompt = prompt_by_id.get(sample_id)\n        if not prompt:\n            missing.append(sample_id)\n            continue\n        response_text = read_response_text(response)\n        parsed_letter = extract_letter(response_text, allowed_letters)\n        is_unparseable = parsed_letter is None\n        is_abstention = detect_abstention(response_text)\n        output.append(\n            {\n                **prompt,\n                \"model_id\": response.get(\"model_id\", \"unknown_model\"),\n                \"provider\": response.get(\"provider\", \"\"),\n                \"model_version\": response.get(\"model_version\", \"\"),\n                \"response_text\": response_text,\n                \"parsed_letter\": parsed_letter or \"\",\n                \"is_unparseable\": is_unparseable,\n                \"is_abstention\": is_abstention,\n                \"is_correct\": (parsed_letter == prompt[\"answer_letter\"]) if parsed_letter else False,\n            }\n        )\n\n    if missing:\n        print(f\"Skipped responses with unknown sample_id: {len(missing)}\")\n    return output\n\n\ndef build_model_summary(rows: list[dict]) -> list[dict]:\n    output = []\n    for model_key, model_rows in grouped(rows, model_key_fn).items():\n        clean_rows = [row for row in model_rows if row[\"family\"] == \"clean\"]\n        real_rows = [row for row in model_rows if row[\"family\"] == \"real_transfer\"]\n        output.append(\n            {\n                **model_key_to_fields(model_key),\n                \"clean_n\": len(clean_rows),\n                \"clean_accuracy\": accuracy(clean_rows),\n                \"clean_unparseable_rate\": rate(clean_rows, \"is_unparseable\"),\n                \"clean_abstention_rate\": rate(clean_rows, \"is_abstention\"),\n                \"real_n\": len(real_rows),\n                \"real_accuracy\": accuracy(real_rows),\n                \"real_unparseable_rate\": rate(real_rows, \"is_unparseable\"),\n                \"real_abstention_rate\": rate(real_rows, \"is_abstention\"),\n                \"accuracy_drop_vs_clean\": accuracy(clean_rows) - accuracy(real_rows),\n            }\n        )\n    return sorted(output, key=lambda row: row[\"model_id\"])\n\n\ndef build_recipe_table(rows: list[dict]) -> list[dict]:\n    output = []\n    for model_key, model_rows in grouped(rows, model_key_fn).items():\n        clean_rows = [row for row in model_rows if row[\"family\"] == \"clean\"]\n        clean_accuracy = accuracy(clean_rows)\n        for recipe, recipe_rows in grouped([row for row in model_rows if row[\"family\"] == \"real_transfer\"], lambda row: row[\"recipe\"]).items():\n            bootstrap = bootstrap_source_matched(clean_rows, recipe_rows, model_key, recipe)\n            output.append(\n                {\n                    **model_key_to_fields(model_key),\n                    \"recipe\": recipe,\n                    \"n\": len(recipe_rows),\n                    \"accuracy\": accuracy(recipe_rows),\n                    \"accuracy_ci_low\": bootstrap[\"real_accuracy_ci_low\"],\n                    \"accuracy_ci_high\": bootstrap[\"real_accuracy_ci_high\"],\n                    \"unparseable_rate\": rate(recipe_rows, \"is_unparseable\"),\n                    \"abstention_rate\": rate(recipe_rows, \"is_abstention\"),\n                    \"accuracy_drop_vs_clean\": clean_accuracy - accuracy(recipe_rows),\n                    \"accuracy_drop_ci_low\": bootstrap[\"accuracy_drop_ci_low\"],\n                    \"accuracy_drop_ci_high\": bootstrap[\"accuracy_drop_ci_high\"],\n                }\n            )\n    return sorted(output, key=lambda row: (row[\"model_id\"], row[\"recipe\"]))\n\n\ndef build_label_table(rows: list[dict]) -> list[dict]:\n    output = []\n    for model_key, model_rows in grouped(rows, model_key_fn).items():\n        labels = sorted({row[\"label\"] for row in model_rows})\n        for label in labels:\n            clean_rows = [row for row in model_rows if row[\"family\"] == \"clean\" and row[\"label\"] == label]\n            real_rows = [row for row in model_rows if row[\"family\"] == \"real_transfer\" and row[\"label\"] == label]\n            output.append(\n                {\n                    **model_key_to_fields(model_key),\n                    \"label\": label,\n                    \"clean_n\": len(clean_rows),\n                    \"clean_accuracy\": accuracy(clean_rows),\n                    \"real_n\": len(real_rows),\n                    \"real_accuracy\": accuracy(real_rows),\n                    \"real_unparseable_rate\": rate(real_rows, \"is_unparseable\"),\n                    \"accuracy_drop_vs_clean\": accuracy(clean_rows) - accuracy(real_rows),\n                }\n            )\n    return sorted(output, key=lambda row: (row[\"model_id\"], float(row[\"real_accuracy\"]), row[\"label\"]))\n\n\ndef build_audit_rows(rows: list[dict]) -> list[dict]:\n    fields = set(AUDIT_FIELDS)\n    return [\n        {key: row[key] for key in AUDIT_FIELDS if key in fields}\n        for row in sorted(rows, key=lambda row: (row[\"model_id\"], row[\"sample_id\"]))\n    ]\n\n\ndef bootstrap_source_matched(clean_rows: list[dict], real_rows: list[dict], model_key: tuple[str, str, str], recipe: str) -> dict[str, float]:\n    source_ids = sorted({row[\"source_selection_id\"] for row in real_rows})\n    clean_by_source = defaultdict(list)\n    real_by_source = defaultdict(list)\n    for row in clean_rows:\n        clean_by_source[row[\"source_selection_id\"]].append(row)\n    for row in real_rows:\n        real_by_source[row[\"source_selection_id\"]].append(row)\n\n    if not source_ids:\n        return {\n            \"real_accuracy_ci_low\": 0.0,\n            \"real_accuracy_ci_high\": 0.0,\n            \"accuracy_drop_ci_low\": 0.0,\n            \"accuracy_drop_ci_high\": 0.0,\n        }\n\n    rng = random.Random(f\"{BOOTSTRAP_SEED}:{model_key}:{recipe}\")\n    real_values = []\n    drop_values = []\n    for _ in range(BOOTSTRAP_SAMPLES):\n        selected = [rng.choice(source_ids) for _ in source_ids]\n        sampled_clean = [row for source_id in selected for row in clean_by_source.get(source_id, [])]\n        sampled_real = [row for source_id in selected for row in real_by_source.get(source_id, [])]\n        clean_acc = accuracy(sampled_clean)\n        real_acc = accuracy(sampled_real)\n        real_values.append(real_acc)\n        drop_values.append(clean_acc - real_acc)\n\n    return {\n        \"real_accuracy_ci_low\": percentile(real_values, 0.025),\n        \"real_accuracy_ci_high\": percentile(real_values, 0.975),\n        \"accuracy_drop_ci_low\": percentile(drop_values, 0.025),\n        \"accuracy_drop_ci_high\": percentile(drop_values, 0.975),\n    }\n\n\ndef read_response_text(row: dict) -> str:\n    for key in [\"response_text\", \"output_text\", \"answer_text\", \"response\", \"content\"]:\n        value = row.get(key)\n        if value is not None:\n            return str(value).strip()\n    return \"\"\n\n\ndef extract_letter(text: str, allowed_letters: list[str]) -> str | None:\n    if not text:\n        return None\n    allowed = \"\".join(re.escape(letter) for letter in allowed_letters)\n    normalized = text.strip().upper()\n    exact = re.fullmatch(rf\"[{allowed}]\", normalized)\n    if exact:\n        return normalized\n    candidates = re.findall(rf\"(?<![A-Z])[{allowed}](?![A-Z])\", normalized)\n    unique = sorted(set(candidates))\n    if len(unique) == 1:\n        return unique[0]\n    first_token = re.match(rf\"^\\s*([{allowed}])[\\).\\s:-]\", normalized)\n    if first_token:\n        return first_token.group(1)\n    return None\n\n\ndef detect_abstention(text: str) -> bool:\n    lowered = text.lower()\n    return any(pattern in lowered for pattern in ABSTENTION_PATTERNS)\n\n\ndef accuracy(rows: list[dict]) -> float:\n    if not rows:\n        return 0.0\n    return sum(1 for row in rows if row[\"is_correct\"]) / len(rows)\n\n\ndef rate(rows: list[dict], key: str) -> float:\n    if not rows:\n        return 0.0\n    return sum(1 for row in rows if row[key]) / len(rows)\n\n\ndef percentile(values: list[float], q: float) -> float:\n    if not values:\n        return 0.0\n    values = sorted(values)\n    index = round((len(values) - 1) * q)\n    return values[index]\n\n\ndef model_key_fn(row: dict) -> tuple[str, str, str]:\n    return (row[\"model_id\"], row.get(\"provider\", \"\"), row.get(\"model_version\", \"\"))\n\n\ndef model_key_to_fields(model_key: tuple[str, str, str]) -> dict[str, str]:\n    model_id, provider, model_version = model_key\n    return {\"model_id\": model_id, \"provider\": provider, \"model_version\": model_version}\n\n\ndef grouped(rows: list[dict], key_fn) -> dict:\n    groups = defaultdict(list)\n    for row in rows:\n        groups[key_fn(row)].append(row)\n    return dict(groups)\n\n\ndef load_jsonl(path: Path) -> list[dict]:\n    rows = []\n    with path.open(\"r\", encoding=\"utf-8\") as handle:\n        for line_number, line in enumerate(handle, start=1):\n            if line.strip():\n                try:\n                    rows.append(json.loads(line))\n                except json.JSONDecodeError as exc:\n                    raise ValueError(f\"Invalid JSON on {path}:{line_number}\") from exc\n    return rows\n\n\ndef write_csv(path: Path, rows: list[dict], fieldnames: list[str]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\"w\", newline=\"\", encoding=\"utf-8\") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fieldnames)\n        writer.writeheader()\n        for row in rows:\n            writer.writerow({key: format_value(row.get(key, \"\")) for key in fieldnames})\n\n\ndef format_value(value) -> str:\n    if isinstance(value, float):\n        return f\"{value:.6f}\"\n    return str(value)\n\n\ndef resolve_project_path(path: str | Path) -> Path:\n    candidate = Path(path)\n    if candidate.is_absolute():\n        return candidate\n    return ROOT / candidate\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n"}

PROMPT_PACK_RELATIVE = Path('reports/vlm_api_track_v03_prompt_pack.jsonl')
MODEL_MATRIX_RELATIVE = Path('configs/vlm_open_weight_model_matrix_v03.json')
DATA_RELATIVE = Path('data')
DATASET_CANDIDATES = [
    Path('/kaggle/input/cure-or-pp-vlm-real-transfer-v03-private'),
    Path('/kaggle/input/datasets/yaroslavkholmirzayev/cure-or-pp-vlm-real-transfer-v03-private'),
    Path('/kaggle/input/cure-or-pp-vlm-real-transfer-v02-private'),
    Path('/kaggle/input/cure-or-pp-vlm-real-transfer-v03-private'),
]

def list_input_roots():
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return []
    return sorted(path for path in input_root.iterdir() if path.is_dir())

def has_data_payload(path):
    return (
        (path / DATA_RELATIVE).exists()
        or (path / 'data.tar').exists()
        or (path / 'data.zip').exists()
    )

def find_data_source_root():
    input_root = Path('/kaggle/input')
    for path in DATASET_CANDIDATES:
        if has_data_payload(path):
            return path
    if input_root.exists():
        payload_candidates = []
        for pattern in ['data', 'data.tar', 'data.zip']:
            payload_candidates.extend(sorted(input_root.rglob(pattern)))
        print('Data payload candidates:', [str(path) for path in payload_candidates[:20]])
        for candidate in payload_candidates:
            root = candidate.parent if candidate.name != 'data' else candidate.parent
            if has_data_payload(root):
                return root
    return None

def materialize_data_root(source_root):
    if (source_root / DATA_RELATIVE).exists():
        return source_root
    unpack_root = Path('/kaggle/working/cure_or_pp_dataset_unpack')
    if unpack_root.exists():
        shutil.rmtree(unpack_root)
    unpack_root.mkdir(parents=True, exist_ok=True)
    data_tar = source_root / 'data.tar'
    data_zip = source_root / 'data.zip'
    if data_tar.exists():
        with tarfile.open(data_tar) as handle:
            handle.extractall(unpack_root)
        return unpack_root
    if data_zip.exists():
        with zipfile.ZipFile(data_zip) as handle:
            handle.extractall(unpack_root)
        return unpack_root
    raise FileNotFoundError(f'No data payload found under {source_root}')

def write_runtime_overrides(data_root):
    runtime_root = Path('/kaggle/working/cure_or_pp_runtime_pack')
    runtime_root.mkdir(parents=True, exist_ok=True)
    for relative_path, text in EMBEDDED_RUNTIME_FILES.items():
        target = runtime_root / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(text, encoding='utf-8')
    runtime_data = runtime_root / DATA_RELATIVE
    if runtime_data.exists() or runtime_data.is_symlink():
        if runtime_data.is_symlink() or runtime_data.is_file():
            runtime_data.unlink()
        else:
            shutil.rmtree(runtime_data)
    runtime_data.symlink_to(data_root / DATA_RELATIVE, target_is_directory=True)
    return runtime_root

print('Python:', sys.version)
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))
try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi not available')
print('Kaggle input roots:', [path.name for path in list_input_roots()])

DATA_SOURCE_ROOT = find_data_source_root()
if DATA_SOURCE_ROOT is None:
    raise FileNotFoundError(
        'Attach the private CURE-OR++ VLM real-transfer dataset to this notebook. '
        f'Checked explicit candidates: {[str(path) for path in DATASET_CANDIDATES]}; '
        f'available /kaggle/input roots: {[path.name for path in list_input_roots()]}'
    )

DATA_ROOT = materialize_data_root(DATA_SOURCE_ROOT)
RUNTIME_ROOT = write_runtime_overrides(DATA_ROOT)
print('DATA_SOURCE_ROOT:', DATA_SOURCE_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('RUNTIME_ROOT:', RUNTIME_ROOT)
print('Prompt pack:', RUNTIME_ROOT / PROMPT_PACK_RELATIVE)
print('Model matrix:', RUNTIME_ROOT / MODEL_MATRIX_RELATIVE)


In [ ]:
%pip install -q --no-cache-dir --force-reinstall --index-url https://download.pytorch.org/whl/cu121 "torch==2.5.1+cu121" "torchvision==0.20.1+cu121"
%pip install -q --no-cache-dir -U "transformers>=4.57,<5" "accelerate>=1.8,<2" "pillow<12" num2words "qwen-vl-utils>=0.0.8,<0.1"


In [ ]:
import transformers
import torch

PROMPT_PACK = DATA_ROOT / PROMPT_PACK_RELATIVE
MODEL_MATRIX_PATH = RUNTIME_ROOT / MODEL_MATRIX_RELATIVE
RUNNER = RUNTIME_ROOT / 'scripts/run_hf_vlm.py'
EVALUATOR = RUNTIME_ROOT / 'scripts/evaluate_vlm_response_pack.py'
PROMPT_PACK = RUNTIME_ROOT / PROMPT_PACK_RELATIVE
OUTPUT_ROOT = Path('/kaggle/working/vlm_open_weight_runs') / RUN_MODE
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for required_path in [PROMPT_PACK, MODEL_MATRIX_PATH, RUNNER, EVALUATOR]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

matrix = json.loads(MODEL_MATRIX_PATH.read_text())
if RUN_MODE not in {'smoke', 'full'}:
    raise ValueError(f'RUN_MODE must be smoke or full, got {RUN_MODE!r}')

all_models = matrix['models']
if SELECTED_MODEL_SLUGS:
    selected = [model for model in all_models if model['slug'] in set(SELECTED_MODEL_SLUGS)]
    missing = sorted(set(SELECTED_MODEL_SLUGS) - {model['slug'] for model in selected})
    if missing:
        raise ValueError(f'Unknown SELECTED_MODEL_SLUGS: {missing}')
else:
    selected = [model for model in all_models if model.get('enabled_by_default', False)]

print('transformers:', transformers.__version__)
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('torch cuda arch list:', torch.cuda.get_arch_list())
    print('torch device capability:', torch.cuda.get_device_capability())

pd.DataFrame(selected)[['slug', 'model_id', 'tier', 'status']]


In [ ]:
def json_arg(value):
    return json.dumps(value or {}, sort_keys=True)

def run_model(model):
    slug = model['slug']
    limit = int(matrix['smoke_limit'] if RUN_MODE == 'smoke' else matrix['full_limit'])
    run_dir = OUTPUT_ROOT / slug
    run_dir.mkdir(parents=True, exist_ok=True)
    responses = run_dir / 'responses.jsonl'
    model_summary = run_dir / 'model_summary.csv'
    recipe_table = run_dir / 'recipe_table.csv'
    label_table = run_dir / 'label_table.csv'
    audit_table = run_dir / 'audit.csv'

    runner_cmd = [
        sys.executable,
        str(RUNNER),
        '--prompt-pack', str(PROMPT_PACK),
        '--provider', model.get('provider', 'huggingface'),
        '--model', model['model_id'],
        '--model-class', model.get('model_class', 'AutoModelForImageTextToText'),
        '--generation-backend', model.get('generation_backend', 'chat_template'),
        '--image-content-key', model.get('image_content_key', 'path'),
        '--output', str(responses),
        '--cache-dir', str(run_dir / 'cache'),
        '--force',
        '--max-tokens', str(model.get('max_tokens', 8)),
        '--dtype', model.get('dtype', 'float16'),
        '--processor-kwargs-json', json_arg(model.get('processor_kwargs')),
        '--model-kwargs-json', json_arg(model.get('model_kwargs')),
        '--generate-kwargs-json', json_arg(model.get('generate_kwargs')),
    ]
    if model.get('revision'):
        runner_cmd.extend(['--revision', model['revision']])
    if model.get('trust_remote_code'):
        runner_cmd.append('--trust-remote-code')
    if model.get('device_map'):
        runner_cmd.extend(['--device-map', model['device_map']])
    if model.get('image_max_side'):
        runner_cmd.extend(['--image-max-side', str(int(model['image_max_side']))])
    if model.get('system_prompt') is not None:
        runner_cmd.extend(['--system-prompt', model['system_prompt']])
    smoke_sample_ids = matrix.get('smoke_sample_ids') if RUN_MODE == 'smoke' else None
    if smoke_sample_ids:
        for sample_id in smoke_sample_ids:
            runner_cmd.extend(['--sample-id', sample_id])
    else:
        runner_cmd.extend(['--limit', str(limit)])

    print('\n=== RUN', slug, model['model_id'], 'limit=', limit, '===')
    print(' '.join(runner_cmd))
    subprocess.run(runner_cmd, check=True, cwd=str(DATA_ROOT))

    eval_cmd = [
        sys.executable,
        str(EVALUATOR),
        '--prompt-pack', str(PROMPT_PACK),
        '--responses', str(responses),
        '--model-summary', str(model_summary),
        '--recipe-table', str(recipe_table),
        '--label-table', str(label_table),
        '--audit-table', str(audit_table),
    ]
    print(' '.join(eval_cmd))
    subprocess.run(eval_cmd, check=True, cwd=str(DATA_ROOT))

    summary = pd.read_csv(model_summary)
    recipe = pd.read_csv(recipe_table)
    label = pd.read_csv(label_table)
    summary.insert(0, 'slug', slug)
    recipe.insert(0, 'slug', slug)
    label.insert(0, 'slug', slug)
    return {'summary': summary, 'recipe': recipe, 'label': label, 'run_dir': run_dir}

results = []
for model in selected:
    try:
        results.append(run_model(model))
    except Exception as exc:
        print(f'FAILED {model["slug"]}: {exc}')
        if RUN_MODE == 'full':
            raise

if not results:
    raise RuntimeError('No successful model runs.')


In [ ]:
summary_df = pd.concat([item['summary'] for item in results], ignore_index=True)
recipe_df = pd.concat([item['recipe'] for item in results], ignore_index=True)
label_df = pd.concat([item['label'] for item in results], ignore_index=True)

summary_out = OUTPUT_ROOT / 'combined_model_summary.csv'
recipe_out = OUTPUT_ROOT / 'combined_recipe_table.csv'
label_out = OUTPUT_ROOT / 'combined_label_table.csv'
summary_df.to_csv(summary_out, index=False)
recipe_df.to_csv(recipe_out, index=False)
label_df.to_csv(label_out, index=False)

manifest = pd.DataFrame([
    {'slug': item['summary']['slug'].iloc[0], 'run_dir': str(item['run_dir'])}
    for item in results
])
manifest_out = OUTPUT_ROOT / 'run_manifest.csv'
manifest.to_csv(manifest_out, index=False)

print('Combined summary:', summary_out)
print('Combined recipe:', recipe_out)
print('Combined labels:', label_out)
print('Manifest:', manifest_out)
summary_df.sort_values(['real_accuracy', 'clean_accuracy'], ascending=False)


In [ ]:
recipe_df.sort_values(['slug', 'recipe'])


In [ ]:
label_df.sort_values(['slug', 'real_accuracy', 'label']).groupby('slug').head(10)
